# AdaptiveMath-AI — Compact Residual-Memory Model and Ablations

**Notebook 05 of 06**

AdaptiveMath-AI combines a histogram-gradient-boosting anchor with Hessian-shrunk question residual memory and a subject-parent fallback for unseen questions. Hyperparameters and calibration are selected with student-grouped nested validation; later temporal and entity-disjoint evaluations remain secondary diagnostics.

## 1. Reproducible environment and metrics

The notebook fixes the random seed, loads the common modeling dataset, defines the probability metrics and creates only runtime model and artifact directories.

In [1]:
from pathlib import Path
import json, math, os, random, time, warnings, tracemalloc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.optimize import minimize, minimize_scalar
from scipy.special import expit, logit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score, average_precision_score, matthews_corrcoef,
    cohen_kappa_score, log_loss, brier_score_loss, confusion_matrix)
import joblib
mpl.rcParams.update({"font.family":"Times New Roman","font.size":18,"axes.titlesize":20,"axes.labelsize":19,
    "xtick.labelsize":18,"ytick.labelsize":18,"legend.fontsize":18,"figure.titlesize":20,
    "axes.spines.top":False,"axes.spines.right":False,"savefig.dpi":350,"figure.dpi":350})
PALETTE={"blue":"#2F6B8F","orange":"#C77C2B","green":"#4F7F3A","purple":"#6D5A8D","red":"#B45A55","gray":"#6E7378","light_gray":"#E7EAED","teal":"#4A8C8A","gold":"#C9A227","black":"#222222","white":"#FFFFFF"}
SEED=20260713; random.seed(SEED); np.random.seed(SEED)
cwd=Path.cwd().resolve(); ROOT=next((p for p in [cwd,*cwd.parents] if (p/"notebooks").is_dir() and (p/"README.md").exists()),None)
if ROOT is None: raise FileNotFoundError("Run from the repository root or the notebooks directory.")
TABLES=ROOT/"artifacts"; TABLES.mkdir(parents=True,exist_ok=True); MODEL_DIR=ROOT/"models/proposed_model_artifacts"; MODEL_DIR.mkdir(parents=True,exist_ok=True)
def safe_p(p): return np.clip(np.asarray(p,dtype=float),1e-6,1-1e-6)
def expected_calibration_error(y,p,bins=15):
    y=np.asarray(y); p=safe_p(p); ids=np.clip(np.digitize(p,np.linspace(0,1,bins+1))-1,0,bins-1); vals=[]
    for b in range(bins):
        m=ids==b
        if m.any(): vals.append((m.mean(),abs(y[m].mean()-p[m].mean())))
    return float(sum(w*g for w,g in vals)),float(max([g for _,g in vals],default=0))
def metric_row(model,y,p,split):
    y=np.asarray(y,dtype=int); p=safe_p(p); pred=(p>=.5).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); ece,mce=expected_calibration_error(y,p)
    try:
        cal=LogisticRegression(C=1e6).fit(logit(p).reshape(-1,1),y); slope=float(cal.coef_[0,0]); intercept=float(cal.intercept_[0])
    except Exception: slope=np.nan; intercept=np.nan
    return {"Model":model,"Split":split,"N":len(y),"Accuracy":accuracy_score(y,pred),"Balanced_Accuracy":balanced_accuracy_score(y,pred),"Macro_F1":f1_score(y,pred,average="macro",zero_division=0),"Weighted_F1":f1_score(y,pred,average="weighted",zero_division=0),"Precision":precision_score(y,pred,zero_division=0),"Recall":recall_score(y,pred,zero_division=0),"Specificity":tn/max(tn+fp,1),"ROC_AUC":roc_auc_score(y,p) if len(np.unique(y))>1 else np.nan,"PR_AUC":average_precision_score(y,p),"MCC":matthews_corrcoef(y,pred),"Kappa":cohen_kappa_score(y,pred),"Log_Loss":log_loss(y,p,labels=[0,1]),"Brier_Score":brier_score_loss(y,p),"ECE":ece,"MCE":mce,"FPR":fp/max(fp+tn,1),"FNR":fn/max(fn+tp,1),"Calibration_Slope":slope,"Calibration_Intercept":intercept,"Confusion_Matrix":json.dumps([[int(tn),int(fp)],[int(fn),int(tp)]])}
def normalize01(s,higher=True):
    s=pd.Series(s,dtype=float); lo,hi=s.min(),s.max(); z=pd.Series(np.ones(len(s)),index=s.index) if not np.isfinite(hi-lo) or hi-lo<1e-12 else (s-lo)/(hi-lo)
    return z if higher else 1-z
print(f"Project root: {ROOT}"); print(f"CPU count: {os.cpu_count()}; seed: {SEED}; selection is validation-only")

Project root: <repository_root>
CPU count: 10; seed: 20260713; selection is validation-only


## 2. Compact residual-memory architecture

An 80-iteration histogram-gradient-boosting anchor is combined with a ridge-regularized one-step Newton item-intercept update. Subject and parent-subject residuals are used only when a question has no learned residual.

In [8]:
data=pd.read_parquet(ROOT/"dataset/processed/modeling_dataset.parquet").reset_index(drop=True)
from hashlib import sha256
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedGroupKFold
from threadpoolctl import threadpool_limits
SEEDS=list(range(20260713,20260723))
split_names=["train","validation","temporal_test","external_unseen_student","external_unseen_question"]
compact_frames={s:data[data.split_id.eq(s)].copy().reset_index(drop=True) for s in split_names}
excluded={"interaction_id","answer_id","user_id","question_id","date_answered","split_id","is_correct","group_id","quiz_id","scheme_of_work_id","session_id","primary_subject_name","academic_term","primary_subject_id","parent_subject_id","gender","premium_pupil","missing_confidence","history_eligible","Confidence","answer_value","correct_answer"}
compact_features=[c for c in data.columns if c not in excluded and pd.api.types.is_numeric_dtype(data[c])]
assert {"missing_confidence","history_eligible","Confidence","answer_value","correct_answer","is_correct"}.isdisjoint(compact_features)
compact_model=Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("model",HistGradientBoostingClassifier(max_iter=80,learning_rate=.1,max_leaf_nodes=31,min_samples_leaf=20,l2_regularization=2,random_state=SEED))])
train_start=time.perf_counter(); compact_model.fit(compact_frames["train"][compact_features],compact_frames["train"].is_correct); compact_train_seconds=time.perf_counter()-train_start
anchor_prob={s:safe_p(compact_model.predict_proba(compact_frames[s][compact_features])[:,1]) for s in split_names if s!="train"}
validation_frame=compact_frames["validation"]; yv=validation_frame.is_correct.to_numpy(dtype=int); validation_groups=validation_frame.user_id.to_numpy()
def fit_newton_memory(frame,y,p,column,shrinkage,cap=.5):
    temp=pd.DataFrame({"key":frame[column].astype(str).to_numpy(),"residual":y-p,"hessian":p*(1-p)})
    agg=temp.groupby("key",dropna=False)[["residual","hessian"]].sum(); correction=(agg.residual/(agg.hessian+shrinkage)).clip(-cap,cap)
    return correction.to_dict(),temp.groupby("key",dropna=False).size().to_dict()
def apply_newton_memory(frame,p,column,memory,fallback_column=None,fallback_memory=None):
    correction=frame[column].astype(str).map(memory)
    if fallback_column is not None and fallback_memory is not None: correction=correction.fillna(frame[fallback_column].astype(str).map(fallback_memory))
    correction=correction.fillna(0).to_numpy(dtype=float)
    return safe_p(expit(logit(safe_p(p))+correction)),correction
def repeated_grouped_oof(stages,seeds=SEEDS):
    predictions=[]; rows=[]
    for seed in seeds:
        out=np.zeros(len(validation_frame)); splitter=StratifiedGroupKFold(5,shuffle=True,random_state=seed)
        for fit_idx,hold_idx in splitter.split(validation_frame,yv,validation_groups):
            fit_frame=validation_frame.iloc[fit_idx]; hold_frame=validation_frame.iloc[hold_idx]; p_fit=anchor_prob["validation"][fit_idx].copy(); p_hold=anchor_prob["validation"][hold_idx].copy()
            for column,shrinkage,cap in stages:
                memory,_=fit_newton_memory(fit_frame,yv[fit_idx],p_fit,column,shrinkage,cap); old_fit=p_fit.copy()
                p_hold,_=apply_newton_memory(hold_frame,p_hold,column,memory); p_fit,_=apply_newton_memory(fit_frame,old_fit,column,memory)
            out[hold_idx]=p_hold
        predictions.append(out); rows.append({"Seed":seed,"ROC_AUC":roc_auc_score(yv,out),"Log_Loss":log_loss(yv,out),"Brier_Score":brier_score_loss(yv,out)})
    return safe_p(np.mean(predictions,axis=0)),pd.DataFrame(rows),predictions
memory_grid=[]; memory_grid_predictions={}; memory_grid_seed_predictions={}
for question_lambda,question_cap in [(2.0,.35),(5.0,.35),(5.0,.50),(10.0,.35),(20.0,.35)]:
 candidate,stability,candidate_seeds=repeated_grouped_oof([("question_id",question_lambda,question_cap)]); memory_grid_predictions[(question_lambda,question_cap)]=candidate; memory_grid_seed_predictions[(question_lambda,question_cap)]=candidate_seeds
 memory_grid.append({"Question_Lambda":question_lambda,"Correction_Cap":question_cap,"Validation_ROC_AUC":roc_auc_score(yv,candidate),"Validation_Log_Loss":log_loss(yv,candidate),"Validation_Brier":brier_score_loss(yv,candidate)})
memory_grid=pd.DataFrame(memory_grid).sort_values(["Validation_Log_Loss","Validation_ROC_AUC"],ascending=[True,False]).reset_index(drop=True); selected_question_lambda=float(memory_grid.iloc[0].Question_Lambda); selected_question_cap=float(memory_grid.iloc[0].Correction_Cap); memory_grid["Selected"]=np.arange(len(memory_grid))==0; memory_grid.to_csv(TABLES/"proposed_model_memory_grid.csv",index=False)
raw_validation=memory_grid_predictions[(selected_question_lambda,selected_question_cap)]; seed_predictions=memory_grid_seed_predictions[(selected_question_lambda,selected_question_cap)]
seed_stability=pd.DataFrame([{"Seed":seed,"ROC_AUC":roc_auc_score(yv,p),"Log_Loss":log_loss(yv,p),"Brier_Score":brier_score_loss(yv,p)} for seed,p in zip(SEEDS,seed_predictions)])
question_only_validation,q_seed_stability,q_seed_predictions=raw_validation,seed_stability.copy(),list(seed_predictions); subject_only_validation,_,s_seed_predictions=repeated_grouped_oof([("primary_subject_id",200.0,.35)])
hgb100_validation=pd.read_parquet(TABLES/"baseline_predictions_validation.parquet"); hgb100_validation=hgb100_validation[hgb100_validation.Model.eq("HistGradientBoosting")].set_index("interaction_id").reindex(validation_frame.interaction_id).Predicted_Probability.to_numpy()
selection_candidates={"HistGradientBoosting (100 trees)":safe_p(hgb100_validation),"Compact HGB anchor (80 trees)":anchor_prob["validation"],"AdaptiveMath-AI":raw_validation}
selection_rows=[]
for name,p in selection_candidates.items():
    evidence="Repeated 5-fold student-grouped validation OOF" if name=="AdaptiveMath-AI" else "Train-only fixed model evaluated on validation"
    ece,_=expected_calibration_error(yv,p); selection_rows.append({"Candidate":name,"Validation_ROC_AUC":roc_auc_score(yv,p),"Validation_Log_Loss":log_loss(yv,p),"Validation_Brier":brier_score_loss(yv,p),"Validation_ECE":ece,"Selection_Data":evidence,"Retained_for_Nested_Evaluation":name=="AdaptiveMath-AI"})
selection=pd.DataFrame(selection_rows).sort_values("Validation_ROC_AUC",ascending=False); selection.to_csv(TABLES/"proposed_model_validation_selection_table.csv",index=False)
seed_stability["Delta_ROC_AUC_vs_HGB100"]=seed_stability.ROC_AUC-roc_auc_score(yv,hgb100_validation); seed_stability.to_csv(TABLES/"proposed_model_seed_stability.csv",index=False)
def compact_fit_calibrator(name,p,y):
    p=safe_p(p)
    if name=="No Calibration": return {"name":name}
    if name=="Platt Scaling": return {"name":name,"model":LogisticRegression(C=1000,max_iter=500,random_state=SEED).fit(logit(p).reshape(-1,1),y)}
    result=minimize_scalar(lambda t:log_loss(y,expit(logit(p)/t),labels=[0,1]),bounds=(.5,2),method="bounded"); return {"name":name,"temperature":float(result.x)}
def compact_apply_calibrator(obj,p):
    p=safe_p(p)
    if obj["name"]=="No Calibration": return p
    if obj["name"]=="Platt Scaling": return safe_p(obj["model"].predict_proba(logit(p).reshape(-1,1))[:,1])
    return safe_p(expit(logit(p)/obj["temperature"]))
calibration_rows=[]; calibration_oof={}
for name in ["No Calibration","Temperature Scaling","Platt Scaling"]:
    out=np.zeros(len(yv)); splitter=StratifiedGroupKFold(5,shuffle=True,random_state=SEED)
    for fit_idx,hold_idx in splitter.split(validation_frame,yv,validation_groups): out[hold_idx]=compact_apply_calibrator(compact_fit_calibrator(name,raw_validation[fit_idx],yv[fit_idx]),raw_validation[hold_idx])
    calibration_oof[name]=safe_p(out); ece,mce=expected_calibration_error(yv,out); calibration_rows.append({"Calibration":name,"Validation_ROC_AUC":roc_auc_score(yv,out),"Validation_Log_Loss":log_loss(yv,out),"Validation_Brier":brier_score_loss(yv,out),"Validation_ECE":ece,"Validation_MCE":mce})
calibration=pd.DataFrame(calibration_rows); eligible=calibration[calibration.Validation_ROC_AUC>=calibration.Validation_ROC_AUC.max()-.0002].copy()
selected_calibration=str(eligible.sort_values(["Validation_Log_Loss","Validation_Brier","Validation_ECE"],ascending=True).iloc[0].Calibration); calibration["Selected"]=calibration.Calibration.eq(selected_calibration); calibration["Selection_Rationale"]="Minimum student-group cross-fitted validation log loss; Brier and ECE are deterministic tie-breakers; AUC must remain within 0.0002 of the maximum"; calibration.to_csv(TABLES/"calibration_summary.csv",index=False)
final_calibrator=compact_fit_calibrator(selected_calibration,raw_validation,yv)
# Item-disjoint grouped validation selects the hierarchical fallback independently of held-out scenarios.
fallback_rows=[]; fallback_predictions={}
for fallback_lambda,fallback_cap in [(50.0,.15),(100.0,.15),(200.0,.15),(200.0,.30),(500.0,.15)]:
 out=np.zeros(len(yv)); splitter=GroupKFold(5)
 for fit_idx,hold_idx in splitter.split(validation_frame,yv,validation_frame.question_id):
  fit_frame=validation_frame.iloc[fit_idx]; hold_frame=validation_frame.iloc[hold_idx]; sm,_=fit_newton_memory(fit_frame,yv[fit_idx],anchor_prob["validation"][fit_idx],"primary_subject_id",fallback_lambda,fallback_cap); pm,_=fit_newton_memory(fit_frame,yv[fit_idx],anchor_prob["validation"][fit_idx],"parent_subject_id",fallback_lambda,fallback_cap); out[hold_idx],_=apply_newton_memory(hold_frame,anchor_prob["validation"][hold_idx],"primary_subject_id",sm,"parent_subject_id",pm)
 fallback_predictions[(fallback_lambda,fallback_cap)]=safe_p(out); fallback_rows.append({"Fallback_Lambda":fallback_lambda,"Fallback_Cap":fallback_cap,"Validation_ROC_AUC":roc_auc_score(yv,out),"Validation_Log_Loss":log_loss(yv,out),"Validation_Brier":brier_score_loss(yv,out),"Grouping":"question_id"})
fallback_selection=pd.DataFrame(fallback_rows).sort_values(["Validation_Log_Loss","Validation_ROC_AUC"],ascending=[True,False]).reset_index(drop=True); selected_fallback_lambda=float(fallback_selection.iloc[0].Fallback_Lambda); selected_fallback_cap=float(fallback_selection.iloc[0].Fallback_Cap); fallback_selection["Selected"]=np.arange(len(fallback_selection))==0; fallback_selection.to_csv(TABLES/"hierarchical_fallback_selection.csv",index=False)
frozen_config={"model_name":"AdaptiveMath-AI","anchor":"HistGradientBoosting-80","subject_memory":{"shrinkage":selected_fallback_lambda,"cap_logit":selected_fallback_cap},"question_memory":{"shrinkage":selected_question_lambda,"cap_logit":selected_question_cap},"hierarchy_backoff":"primary subject -> parent subject -> zero","calibration":selected_calibration,"selection_seeds":SEEDS,"selection_partition":"validation only","holdout_status":"secondary and non-confirmatory; inspected earlier in development"}
frozen_hash=sha256(json.dumps(frozen_config,sort_keys=True).encode()).hexdigest(); print(f"Frozen configuration SHA-256 before holdout evaluation: {frozen_hash}")
subject_memory,subject_counts=fit_newton_memory(validation_frame,yv,anchor_prob["validation"],"primary_subject_id",selected_fallback_lambda,selected_fallback_cap); parent_memory,parent_counts=fit_newton_memory(validation_frame,yv,anchor_prob["validation"],"parent_subject_id",selected_fallback_lambda,selected_fallback_cap)
question_memory,question_counts=fit_newton_memory(validation_frame,yv,anchor_prob["validation"],"question_id",selected_question_lambda,selected_question_cap)
raw_prob={"validation":raw_validation}; correction_parts={"validation":np.zeros((len(yv),2))}; final_prob={"validation":calibration_oof[selected_calibration]}
for split in ["temporal_test","external_unseen_student","external_unseen_question"]:
    frame=compact_frames[split]; qcorr=frame.question_id.astype(str).map(question_memory); scorr=frame.primary_subject_id.astype(str).map(subject_memory).fillna(frame.parent_subject_id.astype(str).map(parent_memory)); correction=qcorr.fillna(scorr).fillna(0).to_numpy(dtype=float); p=safe_p(expit(logit(anchor_prob[split])+correction))
    raw_prob[split]=p; correction_parts[split]=np.c_[scorr.fillna(0).to_numpy(dtype=float),qcorr.fillna(0).to_numpy(dtype=float)]; final_prob[split]=compact_apply_calibrator(final_calibrator,p)
display(selection); display(seed_stability); display(calibration)
print(f"Selected compact architecture: AdaptiveMath-AI; calibration: {selected_calibration}"); print(f"Validation-only frozen hash: {frozen_hash}")

Frozen configuration SHA-256 before holdout evaluation: 8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a5126397674e25b3228861862


,Candidate,Validation_ROC_AUC,Validation_Log_Loss,Validation_Brier,Validation_ECE,Selection_Data,Retained_for_Nested_Evaluation
2,AdaptiveMath-AI,0.777275,0.539720,0.181808,0.010871,Repeated 5-fold student-grouped validation OOF,True
0,HistGradientBoosting (100 trees),0.774307,0.542327,0.182806,0.010775,Train-only fixed model evaluated on validation,False
1,Compact HGB anchor (80 trees),0.774088,0.542498,0.182876,0.010189,Train-only fixed model evaluated on validation,False


,Seed,ROC_AUC,Log_Loss,Brier_Score,Delta_ROC_AUC_vs_HGB100
0,20260713,0.776011,0.541090,0.182311,0.001704
1,20260714,0.776656,0.540440,0.182096,0.002349
2,20260715,0.776444,0.540603,0.182176,0.002137
3,20260716,0.776478,0.540594,0.182156,0.002171
4,20260717,0.776273,0.540768,0.182216,0.001966
5,20260718,0.776697,0.540301,0.182041,0.002390
6,20260719,0.776448,0.540588,0.182132,0.002141
7,20260720,0.776513,0.540481,0.182125,0.002206
8,20260721,0.776912,0.540164,0.181965,0.002605
9,20260722,0.776302,0.540711,0.182220,0.001995


,Calibration,Validation_ROC_AUC,Validation_Log_Loss,Validation_Brier,Validation_ECE,Validation_MCE,Selected,Selection_Rationale
0,No Calibration,0.777275,0.539720,0.181808,0.010871,0.126469,False,Minimum student-group cross-fitted validation ...
1,Temperature Scaling,0.777268,0.539440,0.181721,0.008397,0.035867,False,Minimum student-group cross-fitted validation ...
2,Platt Scaling,0.777255,0.539411,0.181704,0.006613,0.030803,True,Minimum student-group cross-fitted validation ...


Selected compact architecture: AdaptiveMath-AI; calibration: Platt Scaling
Validation-only frozen hash: 8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a5126397674e25b3228861862


## 3. Development configuration, secondary evaluation, and ablations

The whole-validation configuration in this section supplies the frozen model used for secondary temporal and entity-disjoint diagnostics. Its validation values are development estimates, not the primary nested estimate. Component-removal experiments isolate question memory, hierarchical fallback, anchor prediction and calibration; predictive entropy is treated only as a risk-ranking score. Section 5 subsequently replaces the primary validation row with outer-fold predictions from the complete nested tuning procedure.

In [9]:
def crossfit_selected_calibration(p):
    if selected_calibration=="No Calibration": return safe_p(p)
    out=np.zeros(len(yv)); splitter=StratifiedGroupKFold(5,shuffle=True,random_state=SEED)
    for fit_idx,hold_idx in splitter.split(validation_frame,yv,validation_groups):
        out[hold_idx]=compact_apply_calibrator(compact_fit_calibrator(selected_calibration,p[fit_idx],yv[fit_idx]),p[hold_idx])
    return safe_p(out)
q_memory_direct,_=fit_newton_memory(validation_frame,yv,anchor_prob["validation"],"question_id",selected_question_lambda,selected_question_cap)
ablation_prob={s:{} for s in ["validation","temporal_test","external_unseen_student","external_unseen_question"]}
ablation_prob["validation"]={"None (Full AdaptiveMath-AI)":final_prob["validation"],"Hierarchical Subject Residual Memory":crossfit_selected_calibration(question_only_validation),"Question Residual Memory":crossfit_selected_calibration(subject_only_validation),"Compact Multi-View HGB Anchor":np.full(len(yv),yv.mean()),"Calibration":raw_validation}
for split in ["temporal_test","external_unseen_student","external_unseen_question"]:
    frame=compact_frames[split]; q_only,_=apply_newton_memory(frame,anchor_prob[split],"question_id",q_memory_direct); s_only,_=apply_newton_memory(frame,anchor_prob[split],"primary_subject_id",subject_memory,"parent_subject_id",parent_memory)
    ablation_prob[split]={"None (Full AdaptiveMath-AI)":final_prob[split],"Hierarchical Subject Residual Memory":compact_apply_calibrator(final_calibrator,q_only),"Question Residual Memory":compact_apply_calibrator(final_calibrator,s_only),"Compact Multi-View HGB Anchor":np.full(len(frame),yv.mean()),"Calibration":raw_prob[split]}
ablation_interpretation={"None (Full AdaptiveMath-AI)":"Compact multi-view anchor, hierarchical memory, question memory, and calibration.","Hierarchical Subject Residual Memory":"Removes the subject correction and parent-subject cold-start backoff.","Question Residual Memory":"Removes the strongest validation-supported item correction.","Compact Multi-View HGB Anchor":"Removes temporal, psychometric, contextual, hierarchy, and collaborative anchor evidence.","Calibration":"Retains discrimination but removes validation-selected calibration."}
ablation_tables={}
for split,probs in ablation_prob.items():
    rows=[]
    for removed,p in probs.items():
        row=metric_row("None (Full AdaptiveMath-AI)" if removed.startswith("None") else removed,compact_frames[split].is_correct,p,split); row["component_removed"]=removed; row["interpretation"]=ablation_interpretation[removed]; rows.append(row)
    ablation_tables[split]=pd.DataFrame(rows)
ablation_paths={"validation":"ablation_results_validation.csv","temporal_test":"ablation_results_test.csv","external_unseen_student":"ablation_results_external_unseen_student.csv","external_unseen_question":"ablation_results_external_unseen_question.csv"}
for split,name in ablation_paths.items(): ablation_tables[split].to_csv(TABLES/name,index=False)
full_rows={s:t.iloc[0] for s,t in ablation_tables.items()}; contribution=[]
for i,removed in enumerate(ablation_tables["validation"].component_removed):
    contribution.append({"component_removed":removed,"validation_ROC_AUC_change":ablation_tables["validation"].iloc[i].ROC_AUC-full_rows["validation"].ROC_AUC,"test_ROC_AUC_change":ablation_tables["temporal_test"].iloc[i].ROC_AUC-full_rows["temporal_test"].ROC_AUC,"Log_Loss_change":ablation_tables["temporal_test"].iloc[i].Log_Loss-full_rows["temporal_test"].Log_Loss,"Brier_change":ablation_tables["temporal_test"].iloc[i].Brier_Score-full_rows["temporal_test"].Brier_Score,"ECE_change":ablation_tables["temporal_test"].iloc[i].ECE-full_rows["temporal_test"].ECE,"cold_start_student_change":ablation_tables["external_unseen_student"].iloc[i].ROC_AUC-full_rows["external_unseen_student"].ROC_AUC,"cold_start_question_change":ablation_tables["external_unseen_question"].iloc[i].ROC_AUC-full_rows["external_unseen_question"].ROC_AUC,"interpretation":ablation_interpretation[removed]})
contribution=pd.DataFrame(contribution); contribution.to_csv(TABLES/"ablation_component_contribution_summary.csv",index=False)
metrics={}; pred_paths={}; uncertainty_rows=[]
for split,p in final_prob.items():
    frame=compact_frames[split]; y=frame.is_correct.to_numpy(dtype=int); metrics[split]=pd.DataFrame([metric_row("AdaptiveMath-AI",y,p,split)])
    entropy=-(p*np.log(p)+(1-p)*np.log(1-p))/np.log(2); residual_deviation=np.abs(logit(safe_p(p))-logit(safe_p(anchor_prob[split]))); unseen_q=(~frame.question_id.astype(str).isin(question_memory)).astype("int8").to_numpy(); risk=entropy
    out=frame[["interaction_id","answer_id","user_id","question_id","is_correct","split_id","primary_subject_id","primary_subject_name","prior_interaction_count","question_difficulty","subject_mastery_probability"]].copy()
    out["Predicted_Probability_Raw"]=raw_prob[split]; out["Predicted_Probability"]=p; out["Predictive_Entropy"]=entropy; out["Prediction_Risk_Score"]=risk; out["Residual_Deviation"]=residual_deviation; out["Cold_Start_Flag"]=unseen_q
    error=((p>=.5).astype(int)!=y).astype(float); quintile=pd.qcut(pd.Series(risk),5,labels=False,duplicates="drop")
    for level in sorted(pd.unique(quintile)):
        m=np.asarray(quintile==level); uncertainty_rows.append({"Split":split,"Analysis":"Prediction-risk quintile","Level":int(level)+1,"Coverage":m.mean(),"N":int(m.sum()),"Error_Rate":error[m].mean(),"Selective_Accuracy":1-error[m].mean(),"Selective_ROC_AUC":roc_auc_score(y[m],p[m]) if len(np.unique(y[m]))>1 else np.nan,"Mean_Prediction_Risk":risk[m].mean()})
    uncertainty_rows.append({"Split":split,"Analysis":"Error detection","Level":"summary","Coverage":1.0,"N":len(y),"Error_Rate":error.mean(),"Error_Detection_ROC_AUC":roc_auc_score(error,risk) if len(np.unique(error))>1 else np.nan,"Error_Detection_PR_AUC":average_precision_score(error,risk),"Mean_Prediction_Risk":risk.mean()})
    order=np.argsort(risk); risks=[]
    for coverage in np.linspace(.1,1,10):
        idx=order[:max(2,int(len(y)*coverage))]; selective_risk=error[idx].mean(); risks.append(selective_risk); uncertainty_rows.append({"Split":split,"Analysis":"Risk-coverage","Level":coverage,"Coverage":coverage,"N":len(idx),"Error_Rate":selective_risk,"Selective_Accuracy":1-selective_risk,"Selective_ROC_AUC":roc_auc_score(y[idx],p[idx]) if len(np.unique(y[idx]))>1 else np.nan,"Mean_Prediction_Risk":risk[idx].mean()})
    uncertainty_rows.append({"Split":split,"Analysis":"AURC","Level":"summary","Coverage":1.0,"N":len(y),"Error_Rate":np.trapezoid(risks,np.linspace(.1,1,10)),"Selective_Accuracy":np.nan,"Selective_ROC_AUC":np.nan,"Mean_Prediction_Risk":risk.mean()})
    suffix={"validation":"validation","temporal_test":"test","external_unseen_student":"external_unseen_student","external_unseen_question":"external_unseen_question"}[split]; path=TABLES/f"proposed_model_predictions_{suffix}.parquet"; out.to_parquet(path,index=False,compression="zstd"); pred_paths[split]=path; metrics[split].to_csv(TABLES/f"proposed_model_{suffix}_metrics.csv",index=False)
uncertainty_summary=pd.DataFrame(uncertainty_rows); uncertainty_summary.to_csv(TABLES/"uncertainty_quality_summary.csv",index=False)
teacher_review_threshold=float(np.quantile(pd.read_parquet(pred_paths["validation"]).Prediction_Risk_Score,.80))
def assign_compact_decisions(out,threshold=teacher_review_threshold):
    p=out.Predicted_Probability.to_numpy(); u=out.Prediction_Risk_Score.to_numpy(); mastery=pd.to_numeric(out.subject_mastery_probability,errors="coerce").fillna(.5).to_numpy()
    category=np.full(len(out),"Appropriate Reinforcement",dtype=object); action=np.full(len(out),"assign same-level question",dtype=object)
    m=u>=threshold; category[m]="High Prediction Risk"; action[m]="request teacher review"
    m=(u<threshold)&(p>=.85); category[m]="Too Easy"; action[m]="assign harder question"
    m=(u<threshold)&(p>=.55)&(p<.75); category[m]="Productive Challenge"; action[m]="assign same-level question"
    m=(u<threshold)&(p<.40)&(mastery<.45); category[m]="Prerequisite Review"; action[m]="assign easier prerequisite"
    m=(u<threshold)&(p<.50)&(category=="Appropriate Reinforcement"); category[m]="High Risk"; action[m]="provide scaffold"
    return category,action,threshold
decision_rows=[]
for split,path in pred_paths.items():
    out=pd.read_parquet(path); category,action,threshold=assign_compact_decisions(out); out["Risk_Category"]=category; out["Recommended_Action"]=action; out.to_parquet(path,index=False,compression="zstd")
    summary=out.groupby(["split_id","Risk_Category","Recommended_Action"],dropna=False).agg(N=("interaction_id","size"),Mean_Probability=("Predicted_Probability","mean"),Mean_Prediction_Risk=("Prediction_Risk_Score","mean"),Students=("user_id","nunique"),Subjects=("primary_subject_id","nunique")).reset_index(); summary["Proportion"]=summary.N/summary.groupby("split_id").N.transform("sum"); summary["Teacher_Review_Threshold"]=threshold; summary["Threshold_Source"]="validation 80th percentile"; decision_rows.append(summary)
adaptive=pd.concat(decision_rows,ignore_index=True); adaptive.to_csv(TABLES/"adaptive_decision_summary.csv",index=False)
proxy=pd.DataFrame({"Offline_Proxy":["Productive challenge proportion","Student-level coverage","Subject coverage","Recommendation diversity","Teacher-review proportion"],"Value":[adaptive.loc[adaptive.Risk_Category.eq("Productive Challenge"),"N"].sum()/adaptive.N.sum(),adaptive.Students.max()/data.user_id.nunique(),adaptive.Subjects.max()/data.primary_subject_id.nunique(),adaptive.Recommended_Action.nunique(),adaptive.loc[adaptive.Risk_Category.eq("High Prediction Risk"),"N"].sum()/adaptive.N.sum()],"Interpretation":["Predicted—not observed—challenge fit","Research-cohort students represented","Mathematical subjects represented","Distinct actions generated","Predictions routed to human review by a validation-frozen threshold"]}); proxy.to_csv(TABLES/"adaptive_decision_proxy_metrics.csv",index=False)
stress=pd.read_parquet(ROOT/"dataset/processed/stress_short_history.parquet"); ext=pd.read_parquet(pred_paths["external_unseen_student"]); lookup=ext.set_index("answer_id").Predicted_Probability.to_dict(); short_rows=[]
for cap,group in stress.groupby("history_cap"):
    p=np.array([lookup[a] for a in group.answer_id]); short_rows.append({**metric_row("AdaptiveMath-AI",group.is_correct,p,"stress_short_history"),"History_Cap":int(cap)})
short_metrics=pd.DataFrame(short_rows); short_metrics.to_csv(TABLES/"proposed_model_short_history_metrics.csv",index=False)
oof_predictions=pd.DataFrame({"interaction_id":validation_frame.interaction_id,"user_id":validation_frame.user_id,"is_correct":yv,"HGB100":hgb100_validation,"AdaptiveMath_AI":final_prob["validation"],"Minus_Subject_Memory":ablation_prob["validation"]["Hierarchical Subject Residual Memory"],"Minus_Question_Memory":ablation_prob["validation"]["Question Residual Memory"]})
for seed,p,q,s in zip(SEEDS,seed_predictions,q_seed_predictions,s_seed_predictions): oof_predictions[f"AdaptiveMath_AI_seed_{seed}"]=p; oof_predictions[f"Minus_Subject_seed_{seed}"]=q; oof_predictions[f"Minus_Question_seed_{seed}"]=s
oof_predictions.to_parquet(TABLES/"proposed_model_repeated_oof_predictions.parquet",index=False,compression="zstd")
component_registry=pd.DataFrame([
 {"View":"Temporal knowledge state","Compact_Representation":"Shifted rolling, recency, streak, and mastery features in HGB80","Role":"Distilled anchor evidence"},
 {"View":"Hierarchical mathematical concepts","Compact_Representation":"Subject Newton memory with parent-subject cold-start backoff","Role":"Residual correction"},
 {"View":"Psychometric ability-difficulty","Compact_Representation":"Elo, Rasch, BKT, ability-difficulty predictors in HGB80","Role":"Distilled anchor evidence"},
 {"View":"Contextual tabular","Compact_Representation":"80-tree HistGradientBoosting anchor","Role":"Primary nonlinear predictor"},
 {"View":"Collaborative student-question","Compact_Representation":"Matrix-factorization score, similarity, and state norm in HGB80","Role":"Distilled anchor evidence"},
 {"View":"Validation-selected fusion","Compact_Representation":"Hessian-shrunk subject/question Newton memories","Role":"O(log m) sorted-array residual lookup"}]); component_registry.to_csv(TABLES/"proposed_model_component_registry.csv",index=False)
configuration=pd.DataFrame([
 {"Component":"Model name","Selected_Configuration":"AdaptiveMath-AI","Selection_Evidence":"Fixed study name"},
 {"Component":"Compact multi-view anchor","Selected_Configuration":"HistGradientBoosting, 80 trees","Selection_Evidence":"Validation-quality / latency trade-off"},
 {"Component":"Temporal knowledge state","Selected_Configuration":"Leakage-safe historical state distilled in anchor","Selection_Evidence":"Shifted features only"},
 {"Component":"Hierarchical concept memory","Selected_Configuration":"Subject Newton residual λ=200; parent backoff","Selection_Evidence":"Nested search ledger + three-seed removal ablation"},
 {"Component":"Question residual memory","Selected_Configuration":"Question Newton residual λ=5","Selection_Evidence":"Three-seed removal ablation"},
 {"Component":"Psychometric + collaborative evidence","Selected_Configuration":"Compact anchor features","Selection_Evidence":"Train/past-only engineered views"},
 {"Component":"Calibration","Selected_Configuration":selected_calibration,"Selection_Evidence":"Grouped cross-fitted Brier + ECE"},
 {"Component":"Uncertainty","Selected_Configuration":"Entropy + margin + anchor disagreement + cold-start","Selection_Evidence":"Complementary signals"}]); configuration.to_csv(TABLES/"proposed_model_final_configuration.csv",index=False)
display(pd.concat(metrics.values(),ignore_index=True)[["Split","N","ROC_AUC","Log_Loss","Brier_Score","ECE"]]); display(ablation_tables["validation"][["component_removed","ROC_AUC","Log_Loss","Brier_Score","ECE"]]); display(component_registry); display(short_metrics[["History_Cap","N","ROC_AUC","Log_Loss","Brier_Score","ECE"]])
print(f"Saved repeated OOF validation: {TABLES/'proposed_model_repeated_oof_predictions.parquet'}"); print(f"Saved uncertainty quality: {TABLES/'uncertainty_quality_summary.csv'}"); print(f"Saved adaptive decisions: {TABLES/'adaptive_decision_summary.csv'}")

,Split,N,ROC_AUC,Log_Loss,Brier_Score,ECE
0,validation,60750,0.777255,0.539411,0.181704,0.006613
1,temporal_test,63209,0.776305,0.548798,0.185519,0.010316
2,external_unseen_student,22687,0.769423,0.535245,0.180124,0.009380
3,external_unseen_question,21452,0.740980,0.570388,0.194374,0.013031


,component_removed,ROC_AUC,Log_Loss,Brier_Score,ECE
0,None (Full AdaptiveMath-AI),0.777255,0.539411,0.181704,6.613287e-03
1,Hierarchical Subject Residual Memory,0.777255,0.539411,0.181704,6.613287e-03
2,Question Residual Memory,0.774404,0.541781,0.182634,5.341033e-03
3,Compact Multi-View HGB Anchor,0.500000,0.656407,0.231856,3.330669e-16
4,Calibration,0.777275,0.539720,0.181808,1.087075e-02


,View,Compact_Representation,Role
0,Temporal knowledge state,"Shifted rolling, recency, streak, and mastery ...",Distilled anchor evidence
1,Hierarchical mathematical concepts,Subject Newton memory with parent-subject cold...,Residual correction
2,Psychometric ability-difficulty,"Elo, Rasch, BKT, ability-difficulty predictors...",Distilled anchor evidence
3,Contextual tabular,80-tree HistGradientBoosting anchor,Primary nonlinear predictor
4,Collaborative student-question,"Matrix-factorization score, similarity, and st...",Distilled anchor evidence
5,Validation-selected fusion,Hessian-shrunk subject/question Newton memories,O(log m) sorted-array residual lookup


,History_Cap,N,ROC_AUC,Log_Loss,Brier_Score,ECE
0,5,271,0.745416,0.537717,0.181510,0.071309
1,10,271,0.776577,0.523771,0.175333,0.041363
2,20,271,0.776796,0.520155,0.175460,0.053720
3,50,180,0.783917,0.513476,0.168958,0.077592


Saved repeated OOF validation: <repository_root>/artifacts/proposed_model_repeated_oof_predictions.parquet
Saved uncertainty quality: <repository_root>/artifacts/uncertainty_quality_summary.csv
Saved adaptive decisions: <repository_root>/artifacts/adaptive_decision_summary.csv


## 4. Nested equal-label comparison

The complete tuning procedure is estimated with repeated student-grouped outer folds. An HGB100 comparator receives the same label budget, and paired uncertainty is obtained from 5,000 user-cluster bootstrap replicates.

In [10]:
GRID=[(2.0,.35),(5.0,.35),(5.0,.50),(10.0,.35),(20.0,.35)]
FALLBACK_GRID=[(50.0,.15),(100.0,.15),(200.0,.15),(200.0,.30),(500.0,.15)]
def hierarchy_only_prediction(fit_idx,apply_idx,fallback_lambda,fallback_cap):
    fit_frame=validation_frame.iloc[fit_idx]; apply_frame=validation_frame.iloc[apply_idx]; p_fit=anchor_prob["validation"][fit_idx]; p_apply=anchor_prob["validation"][apply_idx]
    subject_memory,_=fit_newton_memory(fit_frame,yv[fit_idx],p_fit,"primary_subject_id",fallback_lambda,fallback_cap); parent_memory,_=fit_newton_memory(fit_frame,yv[fit_idx],p_fit,"parent_subject_id",fallback_lambda,fallback_cap)
    correction=apply_frame.primary_subject_id.astype(str).map(subject_memory).fillna(apply_frame.parent_subject_id.astype(str).map(parent_memory)).fillna(0).to_numpy(dtype=float)
    return safe_p(expit(logit(p_apply)+correction))
def staged_index_prediction(fit_idx,apply_idx,question_lambda,cap,fallback_lambda,fallback_cap):
    fit_frame=validation_frame.iloc[fit_idx]; apply_frame=validation_frame.iloc[apply_idx]; p_fit=anchor_prob["validation"][fit_idx]; p_apply=anchor_prob["validation"][apply_idx]
    question_memory,_=fit_newton_memory(fit_frame,yv[fit_idx],p_fit,"question_id",question_lambda,cap); subject_memory,_=fit_newton_memory(fit_frame,yv[fit_idx],p_fit,"primary_subject_id",fallback_lambda,fallback_cap); parent_memory,_=fit_newton_memory(fit_frame,yv[fit_idx],p_fit,"parent_subject_id",fallback_lambda,fallback_cap)
    question_correction=apply_frame.question_id.astype(str).map(question_memory); fallback=apply_frame.primary_subject_id.astype(str).map(subject_memory).fillna(apply_frame.parent_subject_id.astype(str).map(parent_memory)); correction=question_correction.fillna(fallback).fillna(0).to_numpy(dtype=float)
    return safe_p(expit(logit(p_apply)+correction))
def select_calibration_inside_outer_fit(raw_predictions,labels,groups,seed):
    options={}; rows=[]
    for name in ["No Calibration","Temperature Scaling","Platt Scaling"]:
        calibrated=np.zeros(len(labels)); splitter=StratifiedGroupKFold(4,shuffle=True,random_state=seed)
        for fit_rel,hold_rel in splitter.split(raw_predictions,labels,groups): calibrated[hold_rel]=compact_apply_calibrator(compact_fit_calibrator(name,raw_predictions[fit_rel],labels[fit_rel]),raw_predictions[hold_rel])
        calibrated=safe_p(calibrated); ece,_=expected_calibration_error(labels,calibrated); options[name]=calibrated; rows.append({"Calibration":name,"ROC_AUC":roc_auc_score(labels,calibrated),"Log_Loss":log_loss(labels,calibrated),"Brier":brier_score_loss(labels,calibrated),"ECE":ece})
    scores=pd.DataFrame(rows); eligible=scores[scores.ROC_AUC>=scores.ROC_AUC.max()-.0002]; selected=str(eligible.sort_values(["Log_Loss","Brier","ECE"]).iloc[0].Calibration)
    return selected,options[selected],compact_fit_calibrator(selected,raw_predictions,labels)
FAIR_HGB_INNER_FOLDS=4
def fair_hgb_inner_oof(outer_fit_indices,seed):
    outer_fit_frame=validation_frame.iloc[outer_fit_indices].reset_index(drop=True); labels=outer_fit_frame.is_correct.to_numpy(dtype=int); groups=outer_fit_frame.user_id.to_numpy(); raw_oof=np.zeros(len(outer_fit_frame),dtype=float); filled=np.zeros(len(outer_fit_frame),dtype=bool)
    splitter=StratifiedGroupKFold(FAIR_HGB_INNER_FOLDS,shuffle=True,random_state=seed)
    for inner_fold,(inner_fit,inner_hold) in enumerate(splitter.split(outer_fit_frame,labels,groups),1):
        fair_inner_fit=pd.concat([compact_frames["train"],outer_fit_frame.iloc[inner_fit]],ignore_index=True)
        fair_inner_model=Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("model",HistGradientBoostingClassifier(max_iter=100,learning_rate=.1,max_leaf_nodes=31,min_samples_leaf=20,l2_regularization=2,random_state=seed+inner_fold))])
        fair_inner_model.fit(fair_inner_fit[compact_features],fair_inner_fit.is_correct); raw_oof[inner_hold]=fair_inner_model.predict_proba(outer_fit_frame.iloc[inner_hold][compact_features])[:,1]; filled[inner_hold]=True
    assert filled.all() and np.isfinite(raw_oof).all(); return safe_p(raw_oof),labels,groups
nested_seed_predictions=[]; nested_raw_seed_predictions=[]; fair_validation_seed_predictions=[]; fair_raw_seed_predictions=[]; fair_calibration_names=[]; search_rows=[]
for seed in SEEDS:
    nested_out=np.zeros(len(validation_frame)); nested_raw_out=np.zeros(len(validation_frame)); fair_out=np.zeros(len(validation_frame)); fair_raw_out=np.zeros(len(validation_frame)); outer=StratifiedGroupKFold(5,shuffle=True,random_state=seed)
    for outer_fold,(outer_fit,outer_hold) in enumerate(outer.split(validation_frame,yv,validation_groups),1):
        assert set(validation_groups[outer_fit]).isdisjoint(set(validation_groups[outer_hold])); fallback_scores=[]
        for fallback_lambda,fallback_cap in FALLBACK_GRID:
            fallback_inner=np.zeros(len(outer_fit)); question_inner=GroupKFold(4)
            for fit_rel,hold_rel in question_inner.split(validation_frame.iloc[outer_fit],yv[outer_fit],validation_frame.iloc[outer_fit].question_id): fallback_inner[hold_rel]=hierarchy_only_prediction(outer_fit[fit_rel],outer_fit[hold_rel],fallback_lambda,fallback_cap)
            fallback_scores.append({"Fallback_Lambda":fallback_lambda,"Fallback_Cap":fallback_cap,"Log_Loss":log_loss(yv[outer_fit],fallback_inner),"ROC_AUC":roc_auc_score(yv[outer_fit],fallback_inner)})
        best_fallback=min(fallback_scores,key=lambda row:(row["Log_Loss"],-row["ROC_AUC"])); fallback_lambda=best_fallback["Fallback_Lambda"]; fallback_cap=best_fallback["Fallback_Cap"]; candidate_scores=[]; inner_predictions={}
        for question_lambda,cap in GRID:
            inner_out=np.zeros(len(outer_fit)); inner=StratifiedGroupKFold(4,shuffle=True,random_state=seed+100+outer_fold)
            for inner_fit_rel,inner_hold_rel in inner.split(validation_frame.iloc[outer_fit],yv[outer_fit],validation_groups[outer_fit]): inner_out[inner_hold_rel]=staged_index_prediction(outer_fit[inner_fit_rel],outer_fit[inner_hold_rel],question_lambda,cap,fallback_lambda,fallback_cap)
            inner_predictions[(question_lambda,cap)]=safe_p(inner_out); candidate_scores.append({"ROC_AUC":roc_auc_score(yv[outer_fit],inner_out),"Log_Loss":log_loss(yv[outer_fit],inner_out),"Question_Lambda":question_lambda,"Correction_Cap":cap})
        best=min(candidate_scores,key=lambda row:(row["Log_Loss"],-row["ROC_AUC"])); question_lambda=best["Question_Lambda"]; cap=best["Correction_Cap"]; inner_raw=inner_predictions[(question_lambda,cap)]
        adaptive_calibration,_,adaptive_outer_calibrator=select_calibration_inside_outer_fit(inner_raw,yv[outer_fit],validation_groups[outer_fit],seed+1000+outer_fold); nested_raw_out[outer_hold]=staged_index_prediction(outer_fit,outer_hold,question_lambda,cap,fallback_lambda,fallback_cap); nested_out[outer_hold]=compact_apply_calibrator(adaptive_outer_calibrator,nested_raw_out[outer_hold])
        fair_inner_raw,fair_inner_labels,fair_inner_groups=fair_hgb_inner_oof(outer_fit,seed+2000+outer_fold); fair_calibration,_,fair_outer_calibrator=select_calibration_inside_outer_fit(fair_inner_raw,fair_inner_labels,fair_inner_groups,seed+3000+outer_fold)
        search_rows.append({"Seed":seed,"Outer_Fold":outer_fold,"Fallback_Lambda":fallback_lambda,"Fallback_Cap":fallback_cap,"Fallback_Inner_Log_Loss":best_fallback["Log_Loss"],"Question_Lambda":question_lambda,"Correction_Cap":cap,"Inner_ROC_AUC":best["ROC_AUC"],"Inner_Log_Loss":best["Log_Loss"],"Adaptive_Calibration":adaptive_calibration,"Fair_HGB_Calibration":fair_calibration,"Fair_HGB_Inner_OOF_Folds":FAIR_HGB_INNER_FOLDS,"Student_Group_Overlap":0})
        fair_fit=pd.concat([compact_frames["train"],validation_frame.iloc[outer_fit]],ignore_index=True); fair_fold=Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("model",HistGradientBoostingClassifier(max_iter=100,learning_rate=.1,max_leaf_nodes=31,min_samples_leaf=20,l2_regularization=2,random_state=seed))]); fair_fold.fit(fair_fit[compact_features],fair_fit.is_correct); fair_raw_out[outer_hold]=fair_fold.predict_proba(validation_frame.iloc[outer_hold][compact_features])[:,1]; fair_out[outer_hold]=compact_apply_calibrator(fair_outer_calibrator,fair_raw_out[outer_hold]); fair_calibration_names.append(fair_calibration)
    nested_seed_predictions.append(safe_p(nested_out)); nested_raw_seed_predictions.append(safe_p(nested_raw_out)); fair_validation_seed_predictions.append(safe_p(fair_out)); fair_raw_seed_predictions.append(safe_p(fair_raw_out))
nested_validation_uncalibrated=safe_p(np.mean(nested_raw_seed_predictions,axis=0)); nested_validation=safe_p(np.mean(nested_seed_predictions,axis=0))
fair_validation_raw=safe_p(np.mean(fair_raw_seed_predictions,axis=0)); fair_validation=safe_p(np.mean(fair_validation_seed_predictions,axis=0)); fair_selected_calibration=str(pd.Series(fair_calibration_names).mode().iat[0]); fair_final_calibrator=compact_fit_calibrator(fair_selected_calibration,fair_validation_raw,yv)
search_ledger=pd.DataFrame(search_rows); search_ledger.to_csv(TABLES/"proposed_model_nested_selection_ledger.csv",index=False)
question_mode=search_ledger.groupby(["Question_Lambda","Correction_Cap"]).size().sort_values(ascending=False); fallback_mode=search_ledger.groupby(["Fallback_Lambda","Fallback_Cap"]).size().sort_values(ascending=False)
refit_selection_comparison=pd.DataFrame([{"Component":"Question residual memory","Full_Development_Refit_Selection":f"lambda={selected_question_lambda:g}, cap={selected_question_cap:g}","Modal_Outer_Fit_Selection":f"lambda={question_mode.index[0][0]:g}, cap={question_mode.index[0][1]:g}","Modal_Count":int(question_mode.iloc[0]),"Outer_Fold_Selections":len(search_ledger),"Interpretation":"Nested outer-fold predictions estimate the complete tuning procedure; the serialized refit reselects on all development data and is not claimed to be the identical foldwise configuration."},{"Component":"Hierarchy cold-start fallback","Full_Development_Refit_Selection":f"lambda={selected_fallback_lambda:g}, cap={selected_fallback_cap:g}","Modal_Outer_Fit_Selection":f"lambda={fallback_mode.index[0][0]:g}, cap={fallback_mode.index[0][1]:g}","Modal_Count":int(fallback_mode.iloc[0]),"Outer_Fold_Selections":len(search_ledger),"Interpretation":"Nested outer-fold predictions estimate the complete tuning procedure; the serialized refit reselects on all development data and has no pristine exact-configuration confirmation set."}]); refit_selection_comparison.to_csv(TABLES/"final_refit_selection_comparison.csv",index=False)
nested_predictions=pd.DataFrame({"interaction_id":validation_frame.interaction_id,"user_id":validation_frame.user_id,"is_correct":yv,"AdaptiveMath_AI_nested":nested_validation,"Fair_HGB_nested":fair_validation})
for seed,p,fair_p in zip(SEEDS,nested_seed_predictions,fair_validation_seed_predictions): nested_predictions[f"Adaptive_seed_{seed}"]=p; nested_predictions[f"Fair_HGB_seed_{seed}"]=fair_p
nested_predictions.to_parquet(TABLES/"proposed_model_nested_fair_oof_predictions.parquet",index=False,compression="zstd")
fair_train=pd.concat([compact_frames["train"],compact_frames["validation"]],ignore_index=True); fair_hgb=Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=True)),("model",HistGradientBoostingClassifier(max_iter=100,learning_rate=.1,max_leaf_nodes=31,min_samples_leaf=20,l2_regularization=2,random_state=SEED))]); fair_start=time.perf_counter(); fair_hgb.fit(fair_train[compact_features],fair_train.is_correct); fair_train_seconds=time.perf_counter()-fair_start
fair_predictions={s:compact_apply_calibrator(fair_final_calibrator,safe_p(fair_hgb.predict_proba(compact_frames[s][compact_features])[:,1])) for s in ["temporal_test","external_unseen_student","external_unseen_question"]}; fair_rows=[]
for split,p in fair_predictions.items():
    row=metric_row("HistGradientBoosting (fair train+validation refit)",compact_frames[split].is_correct,p,split); row["AdaptiveMath_AI_ROC_AUC"]=metrics[split].iloc[0].ROC_AUC; row["Delta_ROC_AUC_AdaptiveMinusFairHGB"]=row["AdaptiveMath_AI_ROC_AUC"]-row["ROC_AUC"]; row["Equal_Label_Budget"]=True; fair_rows.append(row)
fair_comparison=pd.DataFrame(fair_rows); fair_comparison["Adaptive_Calibration"]=selected_calibration; fair_comparison["Fair_HGB_Calibration"]=fair_selected_calibration; fair_comparison.to_csv(TABLES/"fair_label_budget_comparison.csv",index=False)
pd.DataFrame({"interaction_id":compact_frames["temporal_test"].interaction_id,"user_id":compact_frames["temporal_test"].user_id,"is_correct":compact_frames["temporal_test"].is_correct,"AdaptiveMath_AI":final_prob["temporal_test"],"Fair_HGB_Refit":fair_predictions["temporal_test"]}).to_parquet(TABLES/"fair_label_budget_predictions_test.parquet",index=False,compression="zstd")
def cluster_bootstrap_difference(frame,p_model,p_comparator,replicates,seed):
    y=frame.is_correct.to_numpy(dtype=int); groups=frame.user_id.to_numpy(); unique=np.unique(groups); index={u:np.flatnonzero(groups==u) for u in unique}; rng=np.random.default_rng(seed); auc_diff=[]; ll_diff=[]; brier_diff=[]
    for _ in range(replicates):
        idx=np.concatenate([index[u] for u in rng.choice(unique,size=len(unique),replace=True)]); yy=y[idx]
        if np.unique(yy).size<2: continue
        auc_diff.append(roc_auc_score(yy,p_model[idx])-roc_auc_score(yy,p_comparator[idx])); ll_diff.append(log_loss(yy,p_model[idx])-log_loss(yy,p_comparator[idx])); brier_diff.append(brier_score_loss(yy,p_model[idx])-brier_score_loss(yy,p_comparator[idx]))
    return np.asarray(auc_diff),np.asarray(ll_diff),np.asarray(brier_diff)
bootstrap_rows=[]
for split,model_p,comp_p in [("validation",nested_validation,fair_validation),("temporal_test",final_prob["temporal_test"],fair_predictions["temporal_test"])]:
    da,dl,db=cluster_bootstrap_difference(compact_frames[split],model_p,comp_p,5000,SEED); bootstrap_p=min(1.0,2*min(((da<=0).sum()+1)/(len(da)+1),((da>=0).sum()+1)/(len(da)+1))); bootstrap_rows.append({"Split":split,"Comparator":"Equal-label-budget HGB100","Replicates":len(da),"Delta_ROC_AUC":roc_auc_score(compact_frames[split].is_correct,model_p)-roc_auc_score(compact_frames[split].is_correct,comp_p),"AUC_CI_Lower":np.quantile(da,.025),"AUC_CI_Upper":np.quantile(da,.975),"AUC_Two_Sided_P":bootstrap_p,"Delta_Log_Loss":log_loss(compact_frames[split].is_correct,model_p)-log_loss(compact_frames[split].is_correct,comp_p),"Log_Loss_CI_Lower":np.quantile(dl,.025),"Log_Loss_CI_Upper":np.quantile(dl,.975),"Delta_Brier":brier_score_loss(compact_frames[split].is_correct,model_p)-brier_score_loss(compact_frames[split].is_correct,comp_p),"Brier_CI_Lower":np.quantile(db,.025),"Brier_CI_Upper":np.quantile(db,.975),"Cluster_Unit":"user_id"})
bootstrap_evidence=pd.DataFrame(bootstrap_rows); bootstrap_evidence["Evidence_Status"]=np.where(bootstrap_evidence.Split.eq("validation"),"selection evidence","secondary non-confirmatory"); bootstrap_evidence.to_csv(TABLES/"proposed_model_cluster_bootstrap.csv",index=False)
# Closest-method comparators isolate the claimed residual-memory contribution under identical grouped folds.
def fit_converged_ridge_item_intercepts(frame,labels,anchor,column="question_id",shrinkage=2.0,max_iter=50,tolerance=1e-8):
 keys=frame[column].astype(str); codes,levels=pd.factorize(keys,sort=True); offsets=logit(safe_p(anchor)); intercepts=np.zeros(len(levels),dtype=float)
 for iteration in range(max_iter):
  probability=expit(offsets+intercepts[codes]); gradient=np.bincount(codes,weights=labels-probability,minlength=len(levels))-shrinkage*intercepts; curvature=np.bincount(codes,weights=probability*(1-probability),minlength=len(levels))+shrinkage; step=gradient/np.maximum(curvature,1e-12); intercepts+=step
  if np.max(np.abs(step))<tolerance: break
 return dict(zip(levels.astype(str),intercepts)),iteration+1
def closest_memory_oof(method,seed):
 out=np.zeros(len(yv)); splitter=StratifiedGroupKFold(5,shuffle=True,random_state=seed)
 for fit_idx,hold_idx in splitter.split(validation_frame,yv,validation_groups):
  fit_frame=validation_frame.iloc[fit_idx]; hold_frame=validation_frame.iloc[hold_idx]; pfit=anchor_prob["validation"][fit_idx]; phold=anchor_prob["validation"][hold_idx]
  if method=="HGB80 anchor": out[hold_idx]=phold; continue
  if method=="Smoothed item target encoding":
   agg=pd.DataFrame({"q":fit_frame.question_id.to_numpy(),"y":yv[fit_idx]}).groupby("q").y.agg(["sum","count"]); mapping=(agg["sum"]+20*yv[fit_idx].mean())/(agg["count"]+20); out[hold_idx]=hold_frame.question_id.map(mapping).fillna(yv[fit_idx].mean()); continue
  if method=="Hierarchical smoothed target encoding":
   global_mean=yv[fit_idx].mean(); qagg=pd.DataFrame({"key":fit_frame.question_id.to_numpy(),"y":yv[fit_idx]}).groupby("key").y.agg(["sum","count"]); sagg=pd.DataFrame({"key":fit_frame.primary_subject_id.to_numpy(),"y":yv[fit_idx]}).groupby("key").y.agg(["sum","count"]); pagg=pd.DataFrame({"key":fit_frame.parent_subject_id.to_numpy(),"y":yv[fit_idx]}).groupby("key").y.agg(["sum","count"]); qmap=(qagg["sum"]+20*global_mean)/(qagg["count"]+20); smap=(sagg["sum"]+50*global_mean)/(sagg["count"]+50); pmap=(pagg["sum"]+100*global_mean)/(pagg["count"]+100); out[hold_idx]=hold_frame.question_id.map(qmap).fillna(hold_frame.primary_subject_id.map(smap)).fillna(hold_frame.parent_subject_id.map(pmap)).fillna(global_mean); continue
  if method=="First-order residual mean":
   temp=pd.DataFrame({"q":fit_frame.question_id.to_numpy(),"r":yv[fit_idx]-pfit}); agg=temp.groupby("q").r.agg(["sum","count"]); corr=agg["sum"]/(agg["count"]+20); out[hold_idx]=safe_p(phold+hold_frame.question_id.map(corr).fillna(0).to_numpy()); continue
  if method=="First-order score correction (matched lambda/cap)":
   temp=pd.DataFrame({"q":fit_frame.question_id.astype(str).to_numpy(),"r":yv[fit_idx]-pfit}); agg=temp.groupby("q").r.agg(["sum","count"]); matched=(agg["sum"]/(agg["count"]+selected_question_lambda)).clip(-selected_question_cap,selected_question_cap); corr=hold_frame.question_id.astype(str).map(matched).fillna(0).to_numpy(dtype=float); out[hold_idx]=safe_p(expit(logit(phold)+corr)); continue
  if method in ["Converged ridge item intercept (offset)","Converged ridge item intercept (matched cap)"]:
   ridge_memory,_=fit_converged_ridge_item_intercepts(fit_frame,yv[fit_idx],pfit,shrinkage=selected_question_lambda); correction=hold_frame.question_id.astype(str).map(ridge_memory).fillna(0).to_numpy(dtype=float); correction=np.clip(correction,-selected_question_cap,selected_question_cap) if method.endswith("(matched cap)") else correction; out[hold_idx]=safe_p(expit(logit(phold)+correction)); continue
  shrink=0.0 if method=="Unshrunk Newton item intercept" else selected_question_lambda; cap=.5 if method=="Unshrunk Newton item intercept" else selected_question_cap; qm,_=fit_newton_memory(fit_frame,yv[fit_idx],pfit,"question_id",shrink,cap)
  if method=="Hessian memory + hierarchy":
   sm,_=fit_newton_memory(fit_frame,yv[fit_idx],pfit,"primary_subject_id",selected_fallback_lambda,selected_fallback_cap); pm,_=fit_newton_memory(fit_frame,yv[fit_idx],pfit,"parent_subject_id",selected_fallback_lambda,selected_fallback_cap); qcorr=hold_frame.question_id.astype(str).map(qm); fallback=hold_frame.primary_subject_id.astype(str).map(sm).fillna(hold_frame.parent_subject_id.astype(str).map(pm)); corr=qcorr.fillna(fallback).fillna(0).to_numpy(); out[hold_idx]=safe_p(expit(logit(phold)+corr))
  else: out[hold_idx],_=apply_newton_memory(hold_frame,phold,"question_id",qm)
 return safe_p(out)
closest_methods=["HGB80 anchor","Smoothed item target encoding","Hierarchical smoothed target encoding","First-order residual mean","First-order score correction (matched lambda/cap)","Unshrunk Newton item intercept","Converged ridge item intercept (offset)","Converged ridge item intercept (matched cap)","Hessian-shrunk item memory","Hessian memory + hierarchy"]; closest_rows=[]; closest_predictions={}
for method in closest_methods:
 predictions_by_seed=[closest_memory_oof(method,seed) for seed in SEEDS]; p=crossfit_selected_calibration(np.mean(predictions_by_seed,axis=0)); closest_predictions[method]=p; closest_rows.append({"Method":method,"Validation_ROC_AUC":roc_auc_score(yv,p),"Validation_Log_Loss":log_loss(yv,p),"Validation_Brier":brier_score_loss(yv,p),"Seeds":len(SEEDS),"Grouping":"student_id","Architecture_Role":"Final architecture" if method=="Hessian memory + hierarchy" else ("Retained core component" if method=="Hessian-shrunk item memory" else "Comparator")})
closest_comparators=pd.DataFrame(closest_rows).sort_values(["Validation_Log_Loss","Validation_ROC_AUC"],ascending=[True,False]); closest_comparators["Evidence_Status"]="Exploratory post-selection mechanism analysis"; closest_comparators.to_csv(TABLES/"residual_memory_closest_comparators.csv",index=False)
closest_prediction_table=pd.DataFrame({"interaction_id":validation_frame.interaction_id,"user_id":validation_frame.user_id,"is_correct":yv}); prediction_columns={}
for index,method in enumerate(closest_methods): column=f"method_{index:02d}"; prediction_columns[method]=column; closest_prediction_table[column]=closest_predictions[method]
closest_prediction_table.to_parquet(TABLES/"residual_memory_closest_comparator_oof_predictions.parquet",index=False,compression="zstd"); reference_method="Hessian memory + hierarchy"; reference_probability=closest_predictions[reference_method]; closest_bootstrap_rows=[]
for index,method in enumerate(closest_methods):
 if method==reference_method: continue
 da,dl,db=cluster_bootstrap_difference(validation_frame,reference_probability,closest_predictions[method],5000,SEED+3000+index); two_sided=lambda values:min(1.0,2*min(((values<=0).sum()+1)/(len(values)+1),((values>=0).sum()+1)/(len(values)+1))); closest_bootstrap_rows.append({"Reference_Method":reference_method,"Comparator":method,"Replicates":len(da),"Delta_ROC_AUC":roc_auc_score(yv,reference_probability)-roc_auc_score(yv,closest_predictions[method]),"AUC_CI_Lower":np.quantile(da,.025),"AUC_CI_Upper":np.quantile(da,.975),"AUC_Two_Sided_P":two_sided(da),"Delta_Log_Loss":log_loss(yv,reference_probability)-log_loss(yv,closest_predictions[method]),"Log_Loss_CI_Lower":np.quantile(dl,.025),"Log_Loss_CI_Upper":np.quantile(dl,.975),"Log_Loss_Two_Sided_P":two_sided(dl),"Delta_Brier":brier_score_loss(yv,reference_probability)-brier_score_loss(yv,closest_predictions[method]),"Brier_CI_Lower":np.quantile(db,.025),"Brier_CI_Upper":np.quantile(db,.975),"Brier_Two_Sided_P":two_sided(db),"Cluster_Unit":"user_id","Prediction_Column":prediction_columns[method],"Evidence_Status":"Exploratory post-selection mechanism analysis"})
def holm_adjusted(pvalues):
 values=np.asarray(pvalues,dtype=float); order=np.argsort(values); adjusted=np.empty(len(values)); running=0.0
 for rank,index in enumerate(order): running=max(running,(len(values)-rank)*values[index]); adjusted[index]=min(running,1.0)
 return adjusted
closest_bootstrap=pd.DataFrame(closest_bootstrap_rows)
for metric in ["AUC","Log_Loss","Brier"]: closest_bootstrap[f"{metric}_Holm_Adjusted_P"]=holm_adjusted(closest_bootstrap[f"{metric}_Two_Sided_P"])
closest_bootstrap["Multiplicity_Family"]="Holm correction across every comparator within each prespecified metric family; AUC, log loss, and Brier families are interpreted separately."
closest_bootstrap["AUC_Supported_After_Holm"]=(closest_bootstrap.AUC_CI_Lower>0)&(closest_bootstrap.AUC_Holm_Adjusted_P<.05); closest_bootstrap["Log_Loss_Supported_After_Holm"]=(closest_bootstrap.Log_Loss_CI_Upper<0)&(closest_bootstrap.Log_Loss_Holm_Adjusted_P<.05); closest_bootstrap["Brier_Supported_After_Holm"]=(closest_bootstrap.Brier_CI_Upper<0)&(closest_bootstrap.Brier_Holm_Adjusted_P<.05); closest_bootstrap.to_csv(TABLES/"residual_memory_closest_comparator_bootstrap.csv",index=False)
matched_reference="Hessian-shrunk item memory"; matched_rows=[]
for index,method in enumerate(["First-order score correction (matched lambda/cap)","Converged ridge item intercept (matched cap)"]):
 da,dl,db=cluster_bootstrap_difference(validation_frame,closest_predictions[matched_reference],closest_predictions[method],5000,SEED+5000+index); two_sided=lambda values:min(1.0,2*min(((values<=0).sum()+1)/(len(values)+1),((values>=0).sum()+1)/(len(values)+1))); matched_rows.append({"Reference_Method":matched_reference,"Comparator":method,"Replicates":len(da),"Delta_ROC_AUC":roc_auc_score(yv,closest_predictions[matched_reference])-roc_auc_score(yv,closest_predictions[method]),"AUC_CI_Lower":np.quantile(da,.025),"AUC_CI_Upper":np.quantile(da,.975),"AUC_Two_Sided_P":two_sided(da),"Delta_Log_Loss":log_loss(yv,closest_predictions[matched_reference])-log_loss(yv,closest_predictions[method]),"Log_Loss_CI_Lower":np.quantile(dl,.025),"Log_Loss_CI_Upper":np.quantile(dl,.975),"Log_Loss_Two_Sided_P":two_sided(dl),"Delta_Brier":brier_score_loss(yv,closest_predictions[matched_reference])-brier_score_loss(yv,closest_predictions[method]),"Brier_CI_Lower":np.quantile(db,.025),"Brier_CI_Upper":np.quantile(db,.975),"Brier_Two_Sided_P":two_sided(db),"Design_Control":"Same HGB80 anchor, student-grouped folds, selected lambda, correction cap, logit application, calibration, seeds, and prediction IDs","Evidence_Status":"Exploratory post-selection matched-mechanism analysis"})
matched_mechanism=pd.DataFrame(matched_rows)
for metric in ["AUC","Log_Loss","Brier"]: matched_mechanism[f"{metric}_Holm_Adjusted_P"]=holm_adjusted(matched_mechanism[f"{metric}_Two_Sided_P"])
matched_mechanism["AUC_Supported_After_Holm"]=(matched_mechanism.AUC_CI_Lower>0)&(matched_mechanism.AUC_Holm_Adjusted_P<.05); matched_mechanism["Log_Loss_Supported_After_Holm"]=(matched_mechanism.Log_Loss_CI_Upper<0)&(matched_mechanism.Log_Loss_Holm_Adjusted_P<.05); matched_mechanism["Brier_Supported_After_Holm"]=(matched_mechanism.Brier_CI_Upper<0)&(matched_mechanism.Brier_Holm_Adjusted_P<.05); matched_mechanism.to_csv(TABLES/"residual_memory_matched_mechanism_bootstrap.csv",index=False)
assert search_ledger.Student_Group_Overlap.eq(0).all(); assert nested_predictions.interaction_id.is_unique and len(nested_predictions)==len(validation_frame); assert nested_predictions.interaction_id.equals(validation_frame.interaction_id); assert np.isfinite(nested_predictions.select_dtypes("number")).all().all()
display(search_ledger); display(closest_comparators); display(closest_bootstrap); display(fair_comparison[["Split","ROC_AUC","AdaptiveMath_AI_ROC_AUC","Delta_ROC_AUC_AdaptiveMinusFairHGB"]]); display(bootstrap_evidence)
print(f"Saved nested selection ledger: {TABLES/'proposed_model_nested_selection_ledger.csv'}"); print(f"Saved cluster bootstrap: {TABLES/'proposed_model_cluster_bootstrap.csv'}")

,Seed,Outer_Fold,Fallback_Lambda,Fallback_Cap,Fallback_Inner_Log_Loss,Question_Lambda,Correction_Cap,Inner_ROC_AUC,Inner_Log_Loss,Adaptive_Calibration,Fair_HGB_Calibration,Fair_HGB_Inner_OOF_Folds,Student_Group_Overlap
0,20260713,1,200.0,0.30,0.543930,5.0,0.35,0.774251,0.542593,Platt Scaling,Platt Scaling,4,0
1,20260713,2,200.0,0.30,0.540275,5.0,0.50,0.778581,0.538369,Platt Scaling,Platt Scaling,4,0
2,20260713,3,100.0,0.15,0.539427,5.0,0.35,0.779467,0.537454,Platt Scaling,Temperature Scaling,4,0
3,20260713,4,200.0,0.30,0.543738,5.0,0.50,0.774549,0.542103,Platt Scaling,Temperature Scaling,4,0
4,20260713,5,100.0,0.15,0.543494,5.0,0.50,0.774541,0.542001,Platt Scaling,Temperature Scaling,4,0
5,20260714,1,100.0,0.15,0.544501,5.0,0.50,0.773586,0.543132,Platt Scaling,Platt Scaling,4,0
6,20260714,2,200.0,0.15,0.541015,5.0,0.50,0.777355,0.539548,Platt Scaling,Temperature Scaling,4,0
7,20260714,3,100.0,0.15,0.540972,5.0,0.50,0.777272,0.539561,Platt Scaling,Temperature Scaling,4,0
8,20260714,4,100.0,0.15,0.542487,5.0,0.50,0.776027,0.540604,Platt Scaling,Platt Scaling,4,0
9,20260714,5,200.0,0.30,0.541974,5.0,0.50,0.776634,0.540111,Platt Scaling,Temperature Scaling,4,0


,Method,Validation_ROC_AUC,Validation_Log_Loss,Validation_Brier,Seeds,Grouping,Architecture_Role,Evidence_Status
9,Hessian memory + hierarchy,0.777273,0.539401,0.181696,10,student_id,Final architecture,Exploratory post-selection mechanism analysis
8,Hessian-shrunk item memory,0.777255,0.539411,0.181704,10,student_id,Retained core component,Exploratory post-selection mechanism analysis
7,Converged ridge item intercept (matched cap),0.777253,0.539413,0.181704,10,student_id,Comparator,Exploratory post-selection mechanism analysis
4,First-order score correction (matched lambda/cap),0.776937,0.539595,0.181756,10,student_id,Comparator,Exploratory post-selection mechanism analysis
6,Converged ridge item intercept (offset),0.776680,0.540086,0.181986,10,student_id,Comparator,Exploratory post-selection mechanism analysis
3,First-order residual mean,0.777054,0.540122,0.181755,10,student_id,Comparator,Exploratory post-selection mechanism analysis
0,HGB80 anchor,0.774065,0.542095,0.182757,10,student_id,Comparator,Exploratory post-selection mechanism analysis
5,Unshrunk Newton item intercept,0.773961,0.542731,0.183073,10,student_id,Comparator,Exploratory post-selection mechanism analysis
2,Hierarchical smoothed target encoding,0.600274,0.641798,0.225117,10,student_id,Comparator,Exploratory post-selection mechanism analysis
1,Smoothed item target encoding,0.598866,0.641838,0.225116,10,student_id,Comparator,Exploratory post-selection mechanism analysis


,Reference_Method,Comparator,Replicates,Delta_ROC_AUC,AUC_CI_Lower,AUC_CI_Upper,AUC_Two_Sided_P,Delta_Log_Loss,Log_Loss_CI_Lower,Log_Loss_CI_Upper,...,Cluster_Unit,Prediction_Column,Evidence_Status,AUC_Holm_Adjusted_P,Log_Loss_Holm_Adjusted_P,Brier_Holm_Adjusted_P,Multiplicity_Family,AUC_Supported_After_Holm,Log_Loss_Supported_After_Holm,Brier_Supported_After_Holm
0,Hessian memory + hierarchy,HGB80 anchor,5000,0.003208,0.002429,0.003981,0.000400,-0.002695,-0.003387,-0.001981,...,user_id,method_00,Exploratory post-selection mechanism analysis,0.003599,0.003599,0.003599,Holm correction across every comparator within...,True,True,True
1,Hessian memory + hierarchy,Smoothed item target encoding,5000,0.178407,0.170684,0.185646,0.000400,-0.102437,-0.107413,-0.097223,...,user_id,method_01,Exploratory post-selection mechanism analysis,0.003599,0.003599,0.003599,Holm correction across every comparator within...,True,True,True
2,Hessian memory + hierarchy,Hierarchical smoothed target encoding,5000,0.176999,0.169413,0.184548,0.000400,-0.102398,-0.107690,-0.097278,...,user_id,method_02,Exploratory post-selection mechanism analysis,0.003599,0.003599,0.003599,Holm correction across every comparator within...,True,True,True
3,Hessian memory + hierarchy,First-order residual mean,5000,0.000219,-0.000101,0.000536,0.171166,-0.000721,-0.001150,-0.000301,...,user_id,method_03,Exploratory post-selection mechanism analysis,0.513497,0.003599,1.000000,Holm correction across every comparator within...,False,True,False
4,Hessian memory + hierarchy,First-order score correction (matched lambda/cap),5000,0.000336,-0.000013,0.000686,0.060788,-0.000195,-0.000513,0.000121,...,user_id,method_04,Exploratory post-selection mechanism analysis,0.243151,0.693461,1.000000,Holm correction across every comparator within...,False,False,False
5,Hessian memory + hierarchy,Unshrunk Newton item intercept,5000,0.003312,0.002672,0.003962,0.000400,-0.003331,-0.003918,-0.002738,...,user_id,method_05,Exploratory post-selection mechanism analysis,0.003599,0.003599,0.003599,Holm correction across every comparator within...,True,True,True
6,Hessian memory + hierarchy,Converged ridge item intercept (offset),5000,0.000594,0.000228,0.000968,0.003199,-0.000685,-0.001022,-0.000339,...,user_id,method_06,Exploratory post-selection mechanism analysis,0.015997,0.003599,0.003599,Holm correction across every comparator within...,True,True,True
7,Hessian memory + hierarchy,Converged ridge item intercept (matched cap),5000,0.000020,-0.000029,0.000071,0.435513,-0.000013,-0.000059,0.000032,...,user_id,method_07,Exploratory post-selection mechanism analysis,0.871026,1.000000,1.000000,Holm correction across every comparator within...,False,False,False
8,Hessian memory + hierarchy,Hessian-shrunk item memory,5000,0.000018,-0.000031,0.000065,0.467506,-0.000011,-0.000053,0.000033,...,user_id,method_08,Exploratory post-selection mechanism analysis,0.871026,1.000000,1.000000,Holm correction across every comparator within...,False,False,False


,Split,ROC_AUC,AdaptiveMath_AI_ROC_AUC,Delta_ROC_AUC_AdaptiveMinusFairHGB
0,temporal_test,0.775174,0.776305,0.001132
1,external_unseen_student,0.769902,0.769423,-0.000479
2,external_unseen_question,0.741171,0.740980,-0.000191


,Split,Comparator,Replicates,Delta_ROC_AUC,AUC_CI_Lower,AUC_CI_Upper,AUC_Two_Sided_P,Delta_Log_Loss,Log_Loss_CI_Lower,Log_Loss_CI_Upper,Delta_Brier,Brier_CI_Lower,Brier_CI_Upper,Cluster_Unit,Evidence_Status
0,validation,Equal-label-budget HGB100,5000,0.001708,0.001077,0.002360,0.000400,-0.001359,-0.001946,-0.000787,-0.000574,-0.000816,-0.000333,user_id,selection evidence
1,temporal_test,Equal-label-budget HGB100,5000,0.001132,0.000197,0.002092,0.017596,-0.000799,-0.001695,0.000074,-0.000317,-0.000687,0.000053,user_id,secondary non-confirmatory


Saved nested selection ledger: <repository_root>/artifacts/proposed_model_nested_selection_ledger.csv
Saved cluster bootstrap: <repository_root>/artifacts/proposed_model_cluster_bootstrap.csv


## 5. Efficiency and model consistency

The selected configuration is serialized, reloaded and compared numerically with its in-memory predictions. One-thread latency, model size, acceptance criteria and the retained-component ablation are recorded with the predictive results.

In [11]:
# Replace canonical validation output with the nested estimate of the selected procedure.
metrics["validation"]=pd.DataFrame([metric_row("AdaptiveMath-AI",yv,nested_validation,"validation")]); metrics["validation"].to_csv(TABLES/"proposed_model_validation_metrics.csv",index=False)
validation_path=TABLES/"proposed_model_predictions_validation.parquet"; validation_out=pd.read_parquet(validation_path); validation_out["Predicted_Probability_Raw"]=nested_validation_uncalibrated; validation_out["Predicted_Probability"]=nested_validation
entropy=-(nested_validation*np.log(nested_validation)+(1-nested_validation)*np.log(1-nested_validation))/np.log(2); validation_out["Predictive_Entropy"]=entropy; validation_out["Prediction_Risk_Score"]=entropy; validation_out["Residual_Deviation"]=np.abs(logit(nested_validation)-logit(anchor_prob["validation"])); validation_out["Cold_Start_Flag"]=(~validation_frame.question_id.astype(str).isin(question_memory)).astype("int8").to_numpy(); validation_out=validation_out.drop(columns=["Probability_Margin_Uncertainty","Expert_Disagreement","Cold_Start_Uncertainty","Uncertainty_Score"],errors="ignore"); validation_out.to_parquet(validation_path,index=False,compression="zstd")
# Freeze one teacher-review threshold on final nested validation predictions and rebuild every decision artifact atomically.
teacher_review_threshold=float(np.quantile(validation_out.Prediction_Risk_Score,.80)); decision_rows=[]; risk_quality_rows=[]
for split,path in pred_paths.items():
 out=validation_out.copy() if split=="validation" else pd.read_parquet(path); category,action,_=assign_compact_decisions(out,teacher_review_threshold); out["Risk_Category"]=category; out["Recommended_Action"]=action; out.to_parquet(path,index=False,compression="zstd")
 summary=out.groupby(["split_id","Risk_Category","Recommended_Action"],dropna=False).agg(N=("interaction_id","size"),Mean_Probability=("Predicted_Probability","mean"),Mean_Prediction_Risk=("Prediction_Risk_Score","mean"),Students=("user_id","nunique"),Subjects=("primary_subject_id","nunique")).reset_index(); summary["Proportion"]=summary.N/summary.groupby("split_id").N.transform("sum"); summary["Teacher_Review_Threshold"]=teacher_review_threshold; summary["Threshold_Source"]="final nested validation 80th percentile"; decision_rows.append(summary)
 yy=out.is_correct.to_numpy(dtype=int); pp=out.Predicted_Probability.to_numpy(dtype=float); rr=out.Prediction_Risk_Score.to_numpy(dtype=float); err=((pp>=.5).astype(int)!=yy).astype(int); quintile=pd.qcut(pd.Series(rr),5,labels=False,duplicates="drop")
 for level in sorted(pd.unique(quintile)):
  mask=np.asarray(quintile==level); risk_quality_rows.append({"Split":split,"Analysis":"Prediction-risk quintile","Level":int(level)+1,"Coverage":mask.mean(),"N":int(mask.sum()),"Error_Rate":err[mask].mean(),"Selective_Accuracy":1-err[mask].mean(),"Selective_ROC_AUC":roc_auc_score(yy[mask],pp[mask]) if len(np.unique(yy[mask]))>1 else np.nan,"Mean_Prediction_Risk":rr[mask].mean()})
 order=np.argsort(rr); coverages=np.linspace(.1,1,10); selective=[]
 for coverage in coverages:
  keep=order[:max(2,int(len(yy)*coverage))]; selective.append(err[keep].mean()); risk_quality_rows.append({"Split":split,"Analysis":"Risk-coverage","Level":coverage,"Coverage":coverage,"N":len(keep),"Error_Rate":err[keep].mean(),"Selective_Accuracy":1-err[keep].mean(),"Selective_ROC_AUC":roc_auc_score(yy[keep],pp[keep]) if len(np.unique(yy[keep]))>1 else np.nan,"Mean_Prediction_Risk":rr[keep].mean()})
 risk_quality_rows.append({"Split":split,"Analysis":"Error detection","Level":"summary","Coverage":1.0,"N":len(yy),"Error_Rate":err.mean(),"Error_Detection_ROC_AUC":roc_auc_score(err,rr) if len(np.unique(err))>1 else np.nan,"Error_Detection_PR_AUC":average_precision_score(err,rr),"Mean_Prediction_Risk":rr.mean()}); risk_quality_rows.append({"Split":split,"Analysis":"AURC","Level":"summary","Coverage":1.0,"N":len(yy),"Error_Rate":np.trapezoid(selective,coverages),"Mean_Prediction_Risk":rr.mean()})
adaptive=pd.concat(decision_rows,ignore_index=True); adaptive.to_csv(TABLES/"adaptive_decision_summary.csv",index=False); pd.DataFrame(risk_quality_rows).to_csv(TABLES/"uncertainty_quality_summary.csv",index=False)

def compact_memory_arrays(memory):
    ordered=sorted((int(float(k)),float(v)) for k,v in memory.items()); return np.asarray([x[0] for x in ordered],dtype=np.int64),np.asarray([x[1] for x in ordered],dtype=np.float32)
question_keys,question_values=compact_memory_arrays(question_memory); subject_keys,subject_values=compact_memory_arrays(subject_memory); parent_keys,parent_values=compact_memory_arrays(parent_memory)
payload={"model_name":"AdaptiveMath-AI","compact_anchor":compact_model,"feature_columns":compact_features,"question_keys":question_keys,"question_values":question_values,"subject_fallback_keys":subject_keys,"subject_fallback_values":subject_values,"parent_fallback_keys":parent_keys,"parent_fallback_values":parent_values,"calibrator":final_calibrator,"configuration":frozen_config,"configuration_sha256":frozen_hash}
model_path=MODEL_DIR/"adaptivemath_ai.joblib"; joblib.dump(payload,model_path,compress=("lzma",9)); config_path=MODEL_DIR/"adaptivemath_ai_configuration.json"; config_path.write_text(json.dumps({**frozen_config,"configuration_sha256":frozen_hash},indent=2),encoding="utf-8")
hgb100_model=joblib.load(ROOT/"models/baseline_model_artifacts/histgradientboosting.joblib")
def fast_memory_lookup(keys,values,query):
    query=np.asarray(query,dtype=np.int64); pos=np.searchsorted(keys,query); valid=pos<len(keys); clipped=np.minimum(pos,len(keys)-1); valid&=keys[clipped].eq(query) if hasattr(keys[clipped],"eq") else keys[clipped]==query; out=np.zeros(len(query),dtype=np.float32); out[valid]=values[clipped[valid]]; return out,valid
def payload_predict(model_payload,frame):
    features=model_payload["feature_columns"]; p=safe_p(model_payload["compact_anchor"].predict_proba(frame[features])[:,1]); qcorr,qknown=fast_memory_lookup(model_payload["question_keys"],model_payload["question_values"],frame.question_id.to_numpy()); scorr,sknown=fast_memory_lookup(model_payload["subject_fallback_keys"],model_payload["subject_fallback_values"],frame.primary_subject_id.to_numpy()); pcorr,pknown=fast_memory_lookup(model_payload["parent_fallback_keys"],model_payload["parent_fallback_values"],frame.parent_subject_id.to_numpy()); fallback=np.where(sknown,scorr,np.where(pknown,pcorr,0)); correction=np.where(qknown,qcorr,fallback); return compact_apply_calibrator(model_payload["calibrator"],safe_p(expit(logit(p)+correction)))
def compact_end_to_end(frame): return payload_predict(payload,frame)
def hgb80_end_to_end(frame): return safe_p(compact_model.predict_proba(frame[compact_features])[:,1])
def hgb100_end_to_end(frame): return safe_p(hgb100_model.predict_proba(frame[compact_features])[:,1])
reloaded_payload=joblib.load(model_path); roundtrip_frame=validation_frame; roundtrip_delta=float(np.max(np.abs(payload_predict(reloaded_payload,roundtrip_frame)-compact_end_to_end(roundtrip_frame)))); assert roundtrip_delta<1e-10; pd.DataFrame([{"Rows":len(roundtrip_frame),"Coverage":"Complete validation partition","Max_Absolute_Probability_Difference":roundtrip_delta,"Status":"PASS"}]).to_csv(TABLES/"model_reload_consistency.csv",index=False)
tracemalloc.start(); _memory_probe=compact_end_to_end(validation_frame); _,python_allocation_peak=tracemalloc.get_traced_memory(); tracemalloc.stop(); del _memory_probe
benchmark_rows=[]
with threadpool_limits(limits=1):
    for batch_size in [1,32,256,4096,len(validation_frame)]:
        batch=validation_frame.iloc[:batch_size]; repeats=200 if batch_size<=256 else 60 if batch_size<10000 else 30
        for _ in range(20): hgb100_end_to_end(batch); hgb80_end_to_end(batch); compact_end_to_end(batch)
        times={"HGB100 comparator":[],"HGB80 anchor":[],"AdaptiveMath-AI":[]}
        for repeat in range(repeats):
            order=["HGB100 comparator","HGB80 anchor","AdaptiveMath-AI"] if repeat%2==0 else ["AdaptiveMath-AI","HGB80 anchor","HGB100 comparator"]
            for name in order:
                start=time.perf_counter(); (hgb100_end_to_end(batch) if name=="HGB100 comparator" else hgb80_end_to_end(batch) if name=="HGB80 anchor" else compact_end_to_end(batch)); times[name].append(time.perf_counter()-start)
        for name,values in times.items():
            arr=np.asarray(values); benchmark_rows.append({"Model":name,"Batch_Size":batch_size,"Repeats":repeats,"Median_Latency_Milliseconds":1000*np.median(arr),"P25_Latency_Milliseconds":1000*np.quantile(arr,.25),"P75_Latency_Milliseconds":1000*np.quantile(arr,.75),"P95_Latency_Milliseconds":1000*np.quantile(arr,.95),"Throughput_Interactions_Per_Second":batch_size/np.median(arr),"Scope":"DataFrame adaptation + imputation + prediction; residual lookup uses sorted arrays and O(log m) search; one CPU thread"})
latency_benchmark=pd.DataFrame(benchmark_rows); latency_benchmark.to_csv(TABLES/"adaptivemath_end_to_end_latency_benchmark.csv",index=False)
full_batch=latency_benchmark[(latency_benchmark.Model.eq("AdaptiveMath-AI"))&(latency_benchmark.Batch_Size.eq(len(validation_frame)))].iloc[0]
latency=pd.DataFrame([{"Model":"AdaptiveMath-AI","Training_Time_Seconds":compact_train_seconds,"Inference_Time_Seconds":full_batch.Median_Latency_Milliseconds/1000,"Evaluated_Interactions":len(validation_frame),"Latency_Milliseconds_Per_Interaction":full_batch.Median_Latency_Milliseconds/len(validation_frame),"Throughput_Interactions_Per_Second":full_batch.Throughput_Interactions_Per_Second,"Model_Size_Bytes":model_path.stat().st_size,"Peak_Memory_Bytes":int(python_allocation_peak),"Memory_Measurement":"tracemalloc peak Python allocations for one full-validation inference; native allocator/RSS excluded","Scope":full_batch.Scope}]); latency.to_csv(TABLES/"latency_efficiency_summary.csv",index=False)
baseline_size=(ROOT/"models/baseline_model_artifacts/histgradientboosting.joblib").stat().st_size
complexity=pd.DataFrame([{"Model":"AdaptiveMath-AI","Expert_Count":1,"Fusion_Complexity_Units":1,"Gradient_Trainable_Parameters_Residual_Layer":0,"Stored_Residual_Memory_Entries":len(question_memory),"Artifact_Size_Bytes":model_path.stat().st_size,"Sequence_Length":0,"Tree_Count":80,"Residual_Memory_Entries":len(question_memory),"Fallback_Memory_Entries":len(subject_memory)+len(parent_memory),"Model_Artifact":str(model_path.relative_to(ROOT))}]); complexity.to_csv(TABLES/"model_complexity_summary.csv",index=False)

# Pruned predictive ablation: only statistically retained components remain.
pruned_ablation={}
for split in ["validation","temporal_test","external_unseen_student","external_unseen_question"]:
    frame=compact_frames[split]; y=frame.is_correct.to_numpy()
    if split=="validation": full_p=nested_validation; no_memory=crossfit_selected_calibration(anchor_prob["validation"])
    else: full_p=final_prob[split]; no_memory=compact_apply_calibrator(final_calibrator,anchor_prob[split])
    rows=[]
    for label,p,meaning in [("None (Full AdaptiveMath-AI)",full_p,"Compact anchor plus statistically retained question residual memory."),("Question Residual Memory",no_memory,"Removes the retained residual branch."),("Compact Multi-View HGB Anchor",np.full(len(frame),yv.mean()),"Removes the compact multi-view anchor.")]:
        row=metric_row("None (Full AdaptiveMath-AI)" if label.startswith("None") else label,y,p,split); row["component_removed"]=label; row["interpretation"]=meaning; rows.append(row)
    pruned_ablation[split]=pd.DataFrame(rows)
for split,name in ablation_paths.items(): pruned_ablation[split].to_csv(TABLES/name,index=False)
cold_start_full=metrics["external_unseen_question"].iloc[0]; cold_start_anchor=metric_row("Without hierarchy fallback",compact_frames["external_unseen_question"].is_correct,compact_apply_calibrator(final_calibrator,anchor_prob["external_unseen_question"]),"external_unseen_question")
cold_start_ablation=pd.DataFrame([{"Variant":"AdaptiveMath-AI with subject-parent fallback","ROC_AUC":cold_start_full.ROC_AUC,"Log_Loss":cold_start_full.Log_Loss,"Brier_Score":cold_start_full.Brier_Score},{"Variant":"Without hierarchy fallback","ROC_AUC":cold_start_anchor["ROC_AUC"],"Log_Loss":cold_start_anchor["Log_Loss"],"Brier_Score":cold_start_anchor["Brier_Score"]}]); cold_start_ablation.to_csv(TABLES/"cold_start_hierarchy_fallback_ablation.csv",index=False)
fullv=pruned_ablation["validation"].iloc[0]; contribution=[]
for i,row in pruned_ablation["validation"].iterrows():
    contribution.append({"component_removed":row.component_removed,"validation_ROC_AUC_change":row.ROC_AUC-fullv.ROC_AUC,"test_ROC_AUC_change":pruned_ablation["temporal_test"].iloc[i].ROC_AUC-pruned_ablation["temporal_test"].iloc[0].ROC_AUC,"Log_Loss_change":pruned_ablation["temporal_test"].iloc[i].Log_Loss-pruned_ablation["temporal_test"].iloc[0].Log_Loss,"Brier_change":pruned_ablation["temporal_test"].iloc[i].Brier_Score-pruned_ablation["temporal_test"].iloc[0].Brier_Score,"ECE_change":pruned_ablation["temporal_test"].iloc[i].ECE-pruned_ablation["temporal_test"].iloc[0].ECE,"cold_start_student_change":pruned_ablation["external_unseen_student"].iloc[i].ROC_AUC-pruned_ablation["external_unseen_student"].iloc[0].ROC_AUC,"cold_start_question_change":pruned_ablation["external_unseen_question"].iloc[i].ROC_AUC-pruned_ablation["external_unseen_question"].iloc[0].ROC_AUC,"interpretation":row.interpretation})
pd.DataFrame(contribution).to_csv(TABLES/"ablation_component_contribution_summary.csv",index=False)

val_adaptive=metric_row("AdaptiveMath-AI",yv,nested_validation,"validation"); val_fair=metric_row("Fair HGB",yv,fair_validation,"validation"); val_boot=bootstrap_evidence[bootstrap_evidence.Split.eq("validation")].iloc[0]; test_boot=bootstrap_evidence[bootstrap_evidence.Split.eq("temporal_test")].iloc[0]; lat=latency_benchmark.pivot(index="Batch_Size",columns="Model",values="Median_Latency_Milliseconds"); p95=latency_benchmark.pivot(index="Batch_Size",columns="Model",values="P95_Latency_Milliseconds")
seed_delta=[roc_auc_score(yv,p)-roc_auc_score(yv,f) for p,f in zip(nested_seed_predictions,fair_validation_seed_predictions)]
gates=[
 ("Selection","Nested fair validation delta AUC >= 0.001",val_adaptive["ROC_AUC"]-val_fair["ROC_AUC"]>=.001,float(val_adaptive["ROC_AUC"]-val_fair["ROC_AUC"])),
 ("Selection","Nested validation AUC CI lower > 0",val_boot.AUC_CI_Lower>0,float(val_boot.AUC_CI_Lower)),
 ("Selection","Nested validation Log Loss delta <= -0.0005",val_adaptive["Log_Loss"]-val_fair["Log_Loss"]<=-.0005,float(val_adaptive["Log_Loss"]-val_fair["Log_Loss"])),
 ("Selection","All ten nested seeds beat fair HGB",min(seed_delta)>0,float(min(seed_delta))),
 ("Selection","Full beats every retained-branch removal",pruned_ablation["validation"].iloc[0].ROC_AUC>pruned_ablation["validation"].iloc[1:].ROC_AUC.max(),float(pruned_ablation["validation"].iloc[0].ROC_AUC-pruned_ablation["validation"].iloc[1:].ROC_AUC.max())),
 ("Secondary diagnostic","Fair temporal delta AUC > 0",test_boot.Delta_ROC_AUC>0,float(test_boot.Delta_ROC_AUC)),
 ("Secondary diagnostic","Fair temporal AUC CI lower > 0",test_boot.AUC_CI_Lower>0,float(test_boot.AUC_CI_Lower)),
 ("Operational","Residual overhead <= 25% at batch 256",lat.loc[256,"AdaptiveMath-AI"]<=1.25*lat.loc[256,"HGB80 anchor"],float(lat.loc[256,"AdaptiveMath-AI"]/lat.loc[256,"HGB80 anchor"])),
 ("Operational","Median latency lower than HGB100 at batch 4096",lat.loc[4096,"AdaptiveMath-AI"]<lat.loc[4096,"HGB100 comparator"],float(lat.loc[4096,"AdaptiveMath-AI"]/lat.loc[4096,"HGB100 comparator"])),
 ("Operational","P95 latency no worse than HGB100 at batch 4096",p95.loc[4096,"AdaptiveMath-AI"]<=p95.loc[4096,"HGB100 comparator"],float(p95.loc[4096,"AdaptiveMath-AI"]/p95.loc[4096,"HGB100 comparator"])),
 ("Operational","Artifact <= 0.80 of HGB100",model_path.stat().st_size<=.8*baseline_size,float(model_path.stat().st_size/baseline_size))]
acceptance=pd.DataFrame(gates,columns=["Gate_Type","Acceptance_Gate","Passed","Observed_Value"]); acceptance["Affects_Model_Acceptance"]=acceptance.Gate_Type.eq("Selection"); acceptance["Configuration_SHA256"]=frozen_hash; acceptance.to_csv(TABLES/"proposed_model_acceptance_gates.csv",index=False); model_accepted=bool(acceptance.loc[acceptance.Affects_Model_Acceptance,"Passed"].all())
assert nested_predictions.interaction_id.is_unique and nested_predictions.interaction_id.equals(validation_frame.interaction_id); assert np.isfinite(nested_predictions.select_dtypes("number")).all().all(); assert frozen_hash==sha256(json.dumps(frozen_config,sort_keys=True).encode()).hexdigest()
configuration=pd.DataFrame([{"Component":"Model name","Selected_Configuration":"AdaptiveMath-AI","Selection_Evidence":"Fixed study label"},{"Component":"Compact anchor","Selected_Configuration":"HGB80","Selection_Evidence":"Nested quality-efficiency protocol"},{"Component":"Question residual memory","Selected_Configuration":f"Newton lambda={selected_question_lambda:g}, cap={selected_question_cap:g}","Selection_Evidence":"Reselected by repeated grouped OOF on all development data for the serialized refit; nested outer predictions estimate the tuning procedure, not this single exact refit configuration"},{"Component":"Subject hierarchy","Selected_Configuration":f"Subject-parent fallback lambda={selected_fallback_lambda:g}, cap={selected_fallback_cap:g}","Selection_Evidence":"Reselected on all development data; cold-start fallback only; exact refit has no pristine confirmation set"},{"Component":"Calibration","Selected_Configuration":selected_calibration,"Selection_Evidence":"Calibration family selected inside each outer fold; full-development refit for secondary deployment diagnostics"},{"Component":"Prediction risk","Selected_Configuration":"Predictive entropy","Selection_Evidence":"Risk-ranking heuristic with error-detection AUROC/AUPRC and AURC"}]); configuration.to_csv(TABLES/"proposed_model_final_configuration.csv",index=False)
display(latency_benchmark); display(cold_start_ablation); display(acceptance)
print(f"Saved model: {model_path} ({model_path.stat().st_size/2**20:.3f} MiB; HGB100 size ratio={model_path.stat().st_size/baseline_size:.3f})")

,Model,Batch_Size,Repeats,Median_Latency_Milliseconds,P25_Latency_Milliseconds,P75_Latency_Milliseconds,P95_Latency_Milliseconds,Throughput_Interactions_Per_Second,Scope
0,HGB100 comparator,1,200,1.699917,1.216292,1.873917,2.405682,588.264185,DataFrame adaptation + imputation + prediction...
1,HGB80 anchor,1,200,1.619979,1.141188,1.803073,2.323921,617.291765,DataFrame adaptation + imputation + prediction...
2,AdaptiveMath-AI,1,200,1.878583,1.403489,2.058615,2.676299,532.315972,DataFrame adaptation + imputation + prediction...
3,HGB100 comparator,32,200,1.884250,1.775323,1.953219,2.161419,16982.884442,DataFrame adaptation + imputation + prediction...
4,HGB80 anchor,32,200,1.809604,1.704771,1.872781,2.109546,17683.426868,DataFrame adaptation + imputation + prediction...
5,AdaptiveMath-AI,32,200,2.101771,2.003198,2.185542,2.508356,15225.255309,DataFrame adaptation + imputation + prediction...
6,HGB100 comparator,256,200,2.934125,2.882958,2.972448,3.193554,87249.179770,DataFrame adaptation + imputation + prediction...
7,HGB80 anchor,256,200,2.755062,2.691198,2.812229,3.105784,92919.868961,DataFrame adaptation + imputation + prediction...
8,AdaptiveMath-AI,256,200,3.150729,3.099897,3.210021,3.606667,81251.037915,DataFrame adaptation + imputation + prediction...
9,HGB100 comparator,4096,60,19.656708,19.456854,20.025521,22.224966,208376.697524,DataFrame adaptation + imputation + prediction...


,Variant,ROC_AUC,Log_Loss,Brier_Score
0,AdaptiveMath-AI with subject-parent fallback,0.740980,0.570388,0.194374
1,Without hierarchy fallback,0.740447,0.570628,0.194465


,Gate_Type,Acceptance_Gate,Passed,Observed_Value,Affects_Model_Acceptance,Configuration_SHA256
0,Selection,Nested fair validation delta AUC >= 0.001,True,0.001708,True,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
1,Selection,Nested validation AUC CI lower > 0,True,0.001077,True,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
2,Selection,Nested validation Log Loss delta <= -0.0005,True,-0.001359,True,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
3,Selection,All ten nested seeds beat fair HGB,True,0.001713,True,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
4,Selection,Full beats every retained-branch removal,True,0.002979,True,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
5,Secondary diagnostic,Fair temporal delta AUC > 0,True,0.001132,False,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
6,Secondary diagnostic,Fair temporal AUC CI lower > 0,True,0.000197,False,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
7,Operational,Residual overhead <= 25% at batch 256,True,1.143615,False,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
8,Operational,Median latency lower than HGB100 at batch 4096,True,0.967862,False,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...
9,Operational,P95 latency no worse than HGB100 at batch 4096,True,0.967415,False,8d9b5bb391cc8fc9b97126bdb3ba756c0b3c1c2a512639...


Saved model: <repository_root>/models/proposed_model_artifacts/adaptivemath_ai.joblib (0.179 MiB; HGB ratio=0.415)
Acceptance status: PASS
